In [3]:
#juan
import pandas as pd
from openpyxl import Workbook
import ast

# Leer el Excel
df = pd.read_excel("Resultado_optimizaciontodasanalisis.xlsx", sheet_name="Sheet1")

# Convertir Fecha a datetime
df["Fecha"] = pd.to_datetime(df["Fecha"], errors="coerce")

# Crear libro y hojas
wb = Workbook()
ws1 = wb.active
ws1.title = "Direcciones Organizadas"
ws2 = wb.create_sheet(title="buscarv")

# Parámetros
column_spacing = 7
row_spacing = 4
listas_por_bloque = 5
headers_inv = ["inv final -50%", "inv final -25%", "inv final 0", "inv final +25%", "inv final +50%"]
headers_lim = ["L-50%", "L -25%", "L 0", "L+25%", "L+50%"]
headers_asi = ["S-50", "S-25", "S", "S+25", "S+50"]
multipliers = [0.5, 0.75, 1.0, 1.25, 1.5]

# === Parte 1: Direcciones Organizadas ===
start_row = 2
start_col = 1

for i in range(0, len(df), listas_por_bloque):
    bloque = df["Estaciones"].iloc[i:i+listas_por_bloque]
    
    for j, item in enumerate(bloque):
        if pd.isna(item):
            continue

        texto = str(item).strip("[]").replace("'", "").replace('"', "")
        direcciones = [x.strip() for x in texto.split(",")]

        col = start_col + j * column_spacing
        ws1.cell(row=start_row - 1, column=col, value="Estacion")

        for k, direccion in enumerate(direcciones):
            ws1.cell(row=start_row + k, column=col, value=direccion)

    max_direcciones = max(len(str(x).strip("[]").split(',')) for x in bloque if pd.notna(x))
    start_row += row_spacing + max_direcciones

# === Parte 2: buscarv (ordenando por columna S dentro de cada bloque) ===
start_row = 3
start_col = 1
tabla_idx = 0

for fecha, grupo in df.groupby(df["Fecha"].dt.date):
    col_offset = tabla_idx * (len(headers_inv) + len(headers_lim) + len(headers_asi) + 6)
    primera_fila = grupo.iloc[0]

    # === Estaciones ===
    estaciones_texto = str(primera_fila["Estaciones"]).strip("[]").replace("'", "").replace('"', "")
    estaciones = [x.strip() for x in estaciones_texto.split(",")]

    data_por_estacion = []

    for _, fila in grupo.iterrows():
        try:
            inventarios = ast.literal_eval(str(fila["Inventario_final"]))
        except:
            inventarios = []

        try:
            asimetria = ast.literal_eval(str(fila["Asimetriamodelo"]))
        except:
            asimetria = []

        try:
            limites = ast.literal_eval(str(fila["Limites"]))
        except:
            limites = []

        try:
            capacidades = ast.literal_eval(str(fila["Capacidad"]))
        except:
            capacidades = []

        for i in range(len(estaciones)):
            estacion = estaciones[i] if i < len(estaciones) else ""
            inv = inventarios[i] if i < len(inventarios) else ""
            asi = asimetria[i] if i < len(asimetria) else 0
            cap = capacidades[i] if i < len(capacidades) else ""
            lim = limites[i] if i < len(limites) else [None, None]

            entry = {
                "Estacion": estacion,
                "inv_final": inv,
                "asimetria": asi,
                "capacidad": cap,
                "limites": lim,
            }

            # Añadir columnas de asimetría ajustadas
            for m, h in zip(multipliers, headers_asi):
                entry[h] = round(asi * m, 2)

            # Añadir columnas de límites ajustadas
            for idx, h in enumerate(headers_lim):
                factor = [0.5, 0.75, 1.0, 1.25, 1.5][idx]
                if isinstance(lim, list) and len(lim) == 2:
                    entry[h] = f"{round(lim[0]*factor, 2)}, {round(lim[1]*factor, 2)}"
                else:
                    entry[h] = ""

            data_por_estacion.append(entry)

    # Ordenar por S de mayor a menor
    data_ordenada = sorted(data_por_estacion, key=lambda x: x["S"], reverse=True)

    # Escribir título de tabla
    ws2.cell(row=start_row - 2, column=start_col + col_offset, value=str(fecha))

    # Escribir encabezados
    encabezados = ["Estacion"] + headers_inv + headers_asi + headers_lim + ["Mmodelo"]
    for idx, encabezado in enumerate(encabezados):
        ws2.cell(row=start_row - 1, column=start_col + col_offset + idx, value=encabezado)

    # Escribir filas ordenadas
    for fila_idx, fila in enumerate(data_ordenada):
        valores = (
            [fila["Estacion"]] +
            [fila["inv_final"]] * len(headers_inv) +
            [fila[h] for h in headers_asi] +
            [fila[h] for h in headers_lim] +
            [fila["capacidad"]]
        )
        for col_idx, val in enumerate(valores):
            ws2.cell(row=start_row + fila_idx, column=start_col + col_offset + col_idx, value=val)

    tabla_idx += 1

# Guardar archivo
wb.save("resultado_completo.xlsx")


In [4]:
import pandas as pd
from openpyxl import Workbook, load_workbook
from openpyxl.utils import get_column_letter
import ast

# === PARTE ORIGINAL (no se modifica nada de esto) ===
df = pd.read_excel("Resultado_optimizaciontodasanalisis.xlsx", sheet_name="Sheet1")
df["Fecha"] = pd.to_datetime(df["Fecha"], errors="coerce")
wb = Workbook()
ws1 = wb.active
ws1.title = "Direcciones Organizadas"
ws2 = wb.create_sheet(title="buscarv")

column_spacing = 7
row_spacing = 4
listas_por_bloque = 5
headers_inv = ["inv final -50%", "inv final -25%", "inv final 0", "inv final +25%", "inv final +50%"]
headers_lim = ["L-50%", "L -25%", "L 0", "L+25%", "L+50%"]
headers_asi = ["S-50", "S-25", "S", "S+25", "S+50"]
multipliers = [0.5, 0.75, 1.0, 1.25, 1.5]

start_row = 2
start_col = 1

for i in range(0, len(df), listas_por_bloque):
    bloque = df["Estaciones"].iloc[i:i+listas_por_bloque]
    for j, item in enumerate(bloque):
        if pd.isna(item):
            continue
        texto = str(item).strip("[]").replace("'", "").replace('"', "")
        direcciones = [x.strip() for x in texto.split(",")]
        col = start_col + j * column_spacing
        ws1.cell(row=start_row - 1, column=col, value="Estacion")
        for k, direccion in enumerate(direcciones):
            ws1.cell(row=start_row + k, column=col, value=direccion)
    max_direcciones = max(len(str(x).strip("[]").split(',')) for x in bloque if pd.notna(x))
    start_row += row_spacing + max_direcciones

start_row = 3
start_col = 1
tabla_idx = 0

for fecha, grupo in df.groupby(df["Fecha"].dt.date):
    col_offset = tabla_idx * (len(headers_inv) + len(headers_lim) + len(headers_asi) + 6)
    primera_fila = grupo.iloc[0]

    estaciones_texto = str(primera_fila["Estaciones"]).strip("[]").replace("'", "").replace('"', "")
    estaciones = [x.strip() for x in estaciones_texto.split(",")]

    ws2.cell(row=start_row - 2, column=start_col + col_offset, value=str(fecha))
    ws2.cell(row=start_row - 1, column=start_col + col_offset, value="Estacion")
    for i, estacion in enumerate(estaciones):
        ws2.cell(row=start_row + i, column=start_col + col_offset, value=estacion)

    for idx, (_, fila) in enumerate(grupo.iterrows()):
        header_inv = headers_inv[idx % len(headers_inv)]
        col_inv = start_col + col_offset + idx + 1
        ws2.cell(row=start_row - 1, column=col_inv, value=header_inv)
        try:
            inventarios = ast.literal_eval(str(fila["Inventario_final"]))
        except:
            inventarios = []
        for j in range(len(estaciones)):
            if j < len(inventarios):
                ws2.cell(row=start_row + j, column=col_inv, value=inventarios[j])

    try:
        asimetria = ast.literal_eval(str(primera_fila["Asimetriamodelo"]))
    except:
        asimetria = []
    for idx, header in enumerate(headers_asi):
        col_asi_var = start_col + col_offset + len(headers_inv) + 1 + idx
        ws2.cell(row=start_row - 1, column=col_asi_var, value=header)
        for j in range(len(estaciones)):
            if j < len(asimetria):
                valor = round(asimetria[j] * multipliers[idx], 2)
                ws2.cell(row=start_row + j, column=col_asi_var, value=valor)

    col_lim_start = start_col + col_offset + len(headers_inv) + len(headers_asi) + 1
    limites_lista = grupo["Limites"].tolist()
    for lim_idx, limites_raw in enumerate(limites_lista):
        header_lim = headers_lim[lim_idx % len(headers_lim)]
        col_lim = col_lim_start + lim_idx
        ws2.cell(row=start_row - 1, column=col_lim, value=header_lim)
        try:
            limites = ast.literal_eval(str(limites_raw))
        except:
            limites = []
        for j in range(len(estaciones)):
            if j < len(limites):
                par = limites[j]
                if isinstance(par, list) and len(par) == 2:
                    valor = f"{round(par[0], 2)}, {round(par[1], 2)}"
                    ws2.cell(row=start_row + j, column=col_lim, value=valor)

    col_columnamo = col_lim + 1
    ws2.cell(row=start_row - 1, column=col_columnamo, value="Mmodelo")
    try:
        capacidades = ast.literal_eval(str(primera_fila["Capacidad"]))
    except:
        capacidades = []
    for j in range(len(estaciones)):
        if j < len(capacidades):
            ws2.cell(row=start_row + j, column=col_columnamo, value=capacidades[j])

    tabla_idx += 1

# === NUEVO: ORDENAR CADA BLOQUE POR LA COLUMNA "S" ===
ws = ws2
max_col = ws.max_column
max_row = ws.max_row

col = 1
while col <= max_col:
    header = ws.cell(row=2, column=col).value
    if header == "Estacion":
        start_col = col
        end_col = col
        while ws.cell(row=2, column=end_col + 1).value != "Mmodelo":
            end_col += 1
        end_col += 1  # incluir columna "Mmodelo"

        # Encontrar la fila base de datos (comienza en 3)
        data_rows = []
        for row in range(3, max_row + 1):
            if ws.cell(row=row, column=start_col).value is None:
                break
            data_rows.append([
                ws.cell(row=row, column=c).value
                for c in range(start_col, end_col + 1)
            ])

        # Buscar índice de columna "S"
        s_col_index = None
        for i in range(start_col, end_col + 1):
            if ws.cell(row=2, column=i).value == "S":
                s_col_index = i - start_col
                break

        if s_col_index is not None:
            # Ordenar por columna S (descendente)
            data_rows.sort(key=lambda x: x[s_col_index] if isinstance(x[s_col_index], (int, float)) else -9999, reverse=True)
            # Sobrescribir
            for i, row_data in enumerate(data_rows):
                for j, val in enumerate(row_data):
                    ws.cell(row=3 + i, column=start_col + j, value=val)

        col = end_col + 1
    else:
        col += 1

# Guardar archivo
wb.save("resultado_completo.xlsx")


In [5]:
# === NUEVO: ORDENAR CADA BLOQUE POR LA COLUMNA "S" ===
ws = ws2
max_col = ws.max_column
max_row = ws.max_row

col = 1
while col <= max_col:
    header = ws.cell(row=2, column=col).value
    if header == "Estacion":
        start_col = col
        end_col = col
        while ws.cell(row=2, column=end_col + 1).value != "Mmodelo":
            end_col += 1
        end_col += 1  # incluir columna "Mmodelo"

        # Encontrar la fila base de datos (comienza en 3)
        data_rows = []
        s_col_index = None

        for i in range(start_col, end_col + 1):
            if ws.cell(row=2, column=i).value == "S":
                s_col_index = i - start_col
                break

        if s_col_index is not None:
            for row in range(3, max_row + 1):
                if ws.cell(row=row, column=start_col).value is None:
                    break
                full_row = []
                for c in range(start_col, end_col + 1):
                    val = ws.cell(row=row, column=c).value
                    full_row.append(val)
                data_rows.append(full_row)

            # Ordenar usando el valor S sin redondear
            data_rows.sort(
                key=lambda x: x[s_col_index] if isinstance(x[s_col_index], (int, float)) else float("-inf"),
                reverse=True
            )

            # Reescribir los datos, mostrando S redondeado pero ordenando con su valor completo
            for i, row_data in enumerate(data_rows):
                for j, val in enumerate(row_data):
                    write_val = round(val, 2) if isinstance(val, float) and ws.cell(row=2, column=start_col + j).value.startswith("S") else val
                    ws.cell(row=3 + i, column=start_col + j, value=write_val)

        col = end_col + 1
    else:
        col += 1


In [6]:
import pandas as pd
from openpyxl import Workbook
import ast

# Leer el Excel
df = pd.read_excel("Resultado_optimizaciontodasanalisis.xlsx", sheet_name="Sheet1")

# Convertir Fecha a datetime
df["Fecha"] = pd.to_datetime(df["Fecha"], errors="coerce")

# Crear libro y hojas
wb = Workbook()
ws1 = wb.active
ws1.title = "Direcciones Organizadas"
ws2 = wb.create_sheet(title="buscarv")

# Parámetros
column_spacing = 7
row_spacing = 4
listas_por_bloque = 5
headers_inv = ["inv final -50%", "inv final -25%", "inv final 0", "inv final +25%", "inv final +50%"]
headers_lim = ["L-50%", "L -25%", "L 0", "L+25%", "L+50%"]
headers_asi = ["S-50", "S-25", "S", "S+25", "S+50"]
multipliers = [0.5, 0.75, 1.0, 1.25, 1.5]

# === Parte 1: Direcciones Organizadas ===
start_row = 2
start_col = 1

for i in range(0, len(df), listas_por_bloque):
    bloque = df["Estaciones"].iloc[i:i+listas_por_bloque]
    
    for j, item in enumerate(bloque):
        if pd.isna(item):
            continue

        texto = str(item).strip("[]").replace("'", "").replace('"', "")
        direcciones = [x.strip() for x in texto.split(",")]

        col = start_col + j * column_spacing
        ws1.cell(row=start_row - 1, column=col, value="Estacion")

        for k, direccion in enumerate(direcciones):
            ws1.cell(row=start_row + k, column=col, value=direccion)

    max_direcciones = max(len(str(x).strip("[]").split(',')) for x in bloque if pd.notna(x))
    start_row += row_spacing + max_direcciones

# === Parte 2: buscarv ===
start_row = 3
start_col = 1
tabla_idx = 0

for fecha, grupo in df.groupby(df["Fecha"].dt.date):
    col_offset = tabla_idx * (len(headers_inv) + len(headers_lim) + len(headers_asi) + 6)
    primera_fila = grupo.iloc[0]

    # === Estaciones ===
    estaciones_texto = str(primera_fila["Estaciones"]).strip("[]").replace("'", "").replace('"', "")
    estaciones = [x.strip() for x in estaciones_texto.split(",")]

    # Encabezado de fecha
    ws2.cell(row=start_row - 2, column=start_col + col_offset, value=str(fecha))
    ws2.cell(row=start_row - 1, column=start_col + col_offset, value="Estacion")

    for i, estacion in enumerate(estaciones):
        ws2.cell(row=start_row + i, column=start_col + col_offset, value=estacion)

    # === Inventarios ===
    for idx, (_, fila) in enumerate(grupo.iterrows()):
        header_inv = headers_inv[idx % len(headers_inv)]
        col_inv = start_col + col_offset + idx + 1
        ws2.cell(row=start_row - 1, column=col_inv, value=header_inv)

        try:
            inventarios = ast.literal_eval(str(fila["Inventario_final"]))
        except:
            inventarios = []

        for j in range(len(estaciones)):
            if j < len(inventarios):
                ws2.cell(row=start_row + j, column=col_inv, value=inventarios[j])

    # === Asimetría y variaciones ===
    try:
        asimetria = ast.literal_eval(str(primera_fila["Asimetriamodelo"]))
    except:
        asimetria = []

    for idx, header in enumerate(headers_asi):
        col_asi_var = start_col + col_offset + len(headers_inv) + 1 + idx
        ws2.cell(row=start_row - 1, column=col_asi_var, value=header)

        for j in range(len(estaciones)):
            if j < len(asimetria):
                valor = asimetria[j] * multipliers[idx]
                if header == "S":
                    ws2.cell(row=start_row + j, column=col_asi_var, value=valor)
                else:
                    ws2.cell(row=start_row + j, column=col_asi_var, value=round(valor, 2))

    # === Límites ===
    col_lim_start = start_col + col_offset + len(headers_inv) + len(headers_asi) + 1
    limites_lista = grupo["Limites"].tolist()

    for lim_idx, limites_raw in enumerate(limites_lista):
        header_lim = headers_lim[lim_idx % len(headers_lim)]
        col_lim = col_lim_start + lim_idx
        ws2.cell(row=start_row - 1, column=col_lim, value=header_lim)

        try:
            limites = ast.literal_eval(str(limites_raw))
        except:
            limites = []

        for j in range(len(estaciones)):
            if j < len(limites):
                par = limites[j]
                if isinstance(par, list) and len(par) == 2:
                    valor = f"{round(par[0], 2)}, {round(par[1], 2)}"
                    ws2.cell(row=start_row + j, column=col_lim, value=valor)

    # === Columnamo ===
    col_columnamo = col_lim + 1
    ws2.cell(row=start_row - 1, column=col_columnamo, value="Mmodelo")

    try:
        capacidades = ast.literal_eval(str(primera_fila["Capacidad"]))
    except:
        capacidades = []

    for j in range(len(estaciones)):
        if j < len(capacidades):
            ws2.cell(row=start_row + j, column=col_columnamo, value=capacidades[j])

    tabla_idx += 1

# === ORDENAR CADA BLOQUE POR "S" DE MAYOR A MENOR, PERO MOSTRAR SOLO 2 DECIMALES ===
ws = ws2
max_col = ws.max_column
max_row = ws.max_row
col = 1

while col <= max_col:
    if ws.cell(row=2, column=col).value == "Estacion":
        start_col = col
        end_col = col
        while ws.cell(row=2, column=end_col + 1).value != "Mmodelo":
            end_col += 1
        end_col += 1

        s_col_index = None
        for i in range(start_col, end_col + 1):
            if ws.cell(row=2, column=i).value == "S":
                s_col_index = i - start_col
                break

        if s_col_index is not None:
            data_rows = []
            for row in range(3, max_row + 1):
                if ws.cell(row=row, column=start_col).value is None:
                    break
                row_data = [ws.cell(row=row, column=c).value for c in range(start_col, end_col + 1)]
                data_rows.append(row_data)

            data_rows.sort(
                key=lambda x: x[s_col_index] if isinstance(x[s_col_index], (int, float)) else float('-inf'),
                reverse=True
            )

            for i, row_data in enumerate(data_rows):
                for j, val in enumerate(row_data):
                    write_val = round(val, 2) if isinstance(val, float) and ws.cell(row=2, column=start_col + j).value.startswith("S") else val
                    ws.cell(row=3 + i, column=start_col + j, value=write_val)

        col = end_col + 1
    else:
        col += 1

# Guardar archivo
wb.save("resultado_completo.xlsx")

In [7]:
import pandas as pd
from openpyxl import Workbook
import ast

# Leer el Excel
df = pd.read_excel("Resultado_optimizaciontodasanalisis.xlsx", sheet_name="Sheet1")

# Convertir Fecha a datetime
df["Fecha"] = pd.to_datetime(df["Fecha"], errors="coerce")

# Crear libro y hojas
wb = Workbook()
ws1 = wb.active
ws1.title = "Direcciones Organizadas"
ws2 = wb.create_sheet(title="buscarv")

# Parámetros
column_spacing = 7
row_spacing = 4
listas_por_bloque = 5
headers_inv = ["inv final -50%", "inv final -25%", "inv final 0", "inv final +25%", "inv final +50%"]
headers_lim = ["L-50%", "L -25%", "L 0", "L+25%", "L+50%"]
headers_asi = ["S-50", "S-25", "S", "S+25", "S+50"]
multipliers = [0.5, 0.75, 1.0, 1.25, 1.5]

# === Parte 1: Direcciones Organizadas ===
start_row = 2
start_col = 1

for i in range(0, len(df), listas_por_bloque):
    bloque = df["Estaciones"].iloc[i:i+listas_por_bloque]
    
    for j, item in enumerate(bloque):
        if pd.isna(item):
            continue

        texto = str(item).strip("[]").replace("'", "").replace('"', "")
        direcciones = [x.strip() for x in texto.split(",")]

        col = start_col + j * column_spacing
        ws1.cell(row=start_row - 1, column=col, value="Estacion")

        for k, direccion in enumerate(direcciones):
            ws1.cell(row=start_row + k, column=col, value=direccion)

    max_direcciones = max(len(str(x).strip("[]").split(',')) for x in bloque if pd.notna(x))
    start_row += row_spacing + max_direcciones

# === Parte 2: buscarv ===
start_row = 3
start_col = 1
tabla_idx = 0

for fecha, grupo in df.groupby(df["Fecha"].dt.date):
    col_offset = tabla_idx * (len(headers_inv) + len(headers_lim) + len(headers_asi) + 7)
    primera_fila = grupo.iloc[0]

    # === Estaciones ===
    estaciones_texto = str(primera_fila["Estaciones"]).strip("[]").replace("'", "").replace('"', "")
    estaciones = [x.strip() for x in estaciones_texto.split(",")]

    # Asimetría pura para ordenamiento
    try:
        asimetria_pura = ast.literal_eval(str(primera_fila["Asimetriamodelo"]))
    except:
        asimetria_pura = []

    # Construir tabla con todos los datos para ordenar por 'S' (asimetría normal)
    tabla = []
    for idx, (_, fila) in enumerate(grupo.iterrows()):
        try:
            inventario_final = ast.literal_eval(str(fila["Inventario_final"]))
        except:
            inventario_final = []

        try:
            limites = ast.literal_eval(str(fila["Limites"]))
        except:
            limites = []

        s_base = asimetria_pura
        if len(s_base) != len(estaciones):
            continue  # Saltar si hay inconsistencia

        tabla.append({
            "inv_final": inventario_final,
            "limites": limites,
            "capacidad": ast.literal_eval(str(fila["Capacidad"])) if pd.notna(fila["Capacidad"]) else [],
            "inv_inicial": ast.literal_eval(str(fila["Inventario_inicial"])) if pd.notna(fila["Inventario_inicial"]) else [],
            "idx": idx,
            "s_base": s_base
        })

    # Ordenar según la asimetría base (S normal), de mayor a menor, sin redondeo
    orden = sorted(range(len(estaciones)), key=lambda i: asimetria_pura[i], reverse=True)

    # Encabezado
    ws2.cell(row=start_row - 2, column=start_col + col_offset, value=str(fecha))
    ws2.cell(row=start_row - 1, column=start_col + col_offset, value="Estacion")
    for i, idx_est in enumerate(orden):
        ws2.cell(row=start_row + i, column=start_col + col_offset, value=estaciones[idx_est])

    # Inventarios
    for inv_idx in range(len(tabla)):
        header_inv = headers_inv[inv_idx % len(headers_inv)]
        col_inv = start_col + col_offset + inv_idx + 1
        ws2.cell(row=start_row - 1, column=col_inv, value=header_inv)
        for i, idx_est in enumerate(orden):
            if idx_est < len(tabla[inv_idx]["inv_final"]):
                ws2.cell(row=start_row + i, column=col_inv, value=tabla[inv_idx]["inv_final"][idx_est])

    # Asimetría y variaciones
    for idx, header in enumerate(headers_asi):
        col_asi_var = start_col + col_offset + len(headers_inv) + 1 + idx
        ws2.cell(row=start_row - 1, column=col_asi_var, value=header)
        for i, idx_est in enumerate(orden):
            if idx_est < len(asimetria_pura):
                valor = asimetria_pura[idx_est] * multipliers[idx]
                ws2.cell(row=start_row + i, column=col_asi_var, value=round(valor, 2))

    # Límites
    col_lim_start = start_col + col_offset + len(headers_inv) + len(headers_asi) + 1
    for lim_idx in range(len(tabla)):
        header_lim = headers_lim[lim_idx % len(headers_lim)]
        col_lim = col_lim_start + lim_idx
        ws2.cell(row=start_row - 1, column=col_lim, value=header_lim)
        for i, idx_est in enumerate(orden):
            if idx_est < len(tabla[lim_idx]["limites"]):
                par = tabla[lim_idx]["limites"][idx_est]
                if isinstance(par, list) and len(par) == 2:
                    valor = f"{round(par[0], 2)}, {round(par[1], 2)}"
                    ws2.cell(row=start_row + i, column=col_lim, value=valor)

    # B inicial
    col_binicial = col_lim + 1
    ws2.cell(row=start_row - 1, column=col_binicial, value="B inicial")
    for i, idx_est in enumerate(orden):
        if idx_est < len(tabla[0]["inv_inicial"]):
            ws2.cell(row=start_row + i, column=col_binicial, value=tabla[0]["inv_inicial"][idx_est])

    # Mmodelo
    col_columnamo = col_binicial + 1
    ws2.cell(row=start_row - 1, column=col_columnamo, value="Mmodelo")
    for i, idx_est in enumerate(orden):
        if idx_est < len(tabla[0]["capacidad"]):
            ws2.cell(row=start_row + i, column=col_columnamo, value=tabla[0]["capacidad"][idx_est])

    tabla_idx += 1

# Guardar archivo
wb.save("resultado_completo.xlsx")


In [16]:
import pandas as pd
from openpyxl import Workbook
import ast

# Leer el Excel
df = pd.read_excel("Resultado_optimizaciontodasanalisis.xlsx", sheet_name="Sheet1")

# Convertir Fecha a datetime
df["Fecha"] = pd.to_datetime(df["Fecha"], errors="coerce")

# Crear libro y hojas
wb = Workbook()
ws1 = wb.active
ws1.title = "Direcciones Organizadas"
ws2 = wb.create_sheet(title="buscarv")

# Parámetros
column_spacing = 7
row_spacing = 4
listas_por_bloque = 5
headers_inv = ["inv final -50%", "inv final -25%", "inv final 0", "inv final +25%", "inv final +50%"]
headers_lim = ["L-50%", "L -25%", "L 0", "L+25%", "L+50%"]
headers_asi = ["S-50", "S-25", "S", "S+25", "S+50"]
multipliers = [0.5, 0.75, 1.0, 1.25, 1.5]

# === Parte 1: Direcciones Organizadas ===
start_row = 2
start_col = 1

for i in range(0, len(df), listas_por_bloque):
    bloque = df["Estaciones"].iloc[i:i+listas_por_bloque]
    
    for j, item in enumerate(bloque):
        if pd.isna(item):
            continue

        texto = str(item).strip("[]").replace("'", "").replace('"', "")
        direcciones = [x.strip() for x in texto.split(",")]

        col = start_col + j * column_spacing
        ws1.cell(row=start_row - 1, column=col, value="Estacion")

        for k, direccion in enumerate(direcciones):
            ws1.cell(row=start_row + k, column=col, value=direccion)

    max_direcciones = max(len(str(x).strip("[]").split(',')) for x in bloque if pd.notna(x))
    start_row += row_spacing + max_direcciones

# === Parte 2: buscarv ===
start_row = 3
start_col = 1
tabla_idx = 0

for fecha, grupo in df.groupby(df["Fecha"].dt.date):
    col_offset = tabla_idx * (len(headers_inv) + len(headers_lim) + len(headers_asi) + 7 + len(headers_inv))  # + len(headers_inv) para columnas V-xx nuevas
    primera_fila = grupo.iloc[0]

    # === Estaciones ===
    estaciones_texto = str(primera_fila["Estaciones"]).strip("[]").replace("'", "").replace('"', "")
    estaciones = [x.strip() for x in estaciones_texto.split(",")]

    # Asimetría pura para ordenamiento
    try:
        asimetria_pura = ast.literal_eval(str(primera_fila["Asimetriamodelo"]))
    except:
        asimetria_pura = []

    # Construir tabla con todos los datos para ordenar por 'S' (asimetría normal)
    tabla = []
    for idx, (_, fila) in enumerate(grupo.iterrows()):
        try:
            inventario_final = ast.literal_eval(str(fila["Inventario_final"]))
        except:
            inventario_final = []

        try:
            limites = ast.literal_eval(str(fila["Limites"]))
        except:
            limites = []

        s_base = asimetria_pura
        if len(s_base) != len(estaciones):
            continue  # Saltar si hay inconsistencia

        tabla.append({
            "inv_final": inventario_final,
            "limites": limites,
            "capacidad": ast.literal_eval(str(fila["Capacidad"])) if pd.notna(fila["Capacidad"]) else [],
            "inv_inicial": ast.literal_eval(str(fila["Inventario_inicial"])) if pd.notna(fila["Inventario_inicial"]) else [],
            "idx": idx,
            "s_base": s_base
        })

    # Ordenar según la asimetría base (S normal), de mayor a menor, sin redondeo
    orden = sorted(range(len(estaciones)), key=lambda i: asimetria_pura[i], reverse=True)

    # Encabezado
    ws2.cell(row=start_row - 2, column=start_col + col_offset, value=str(fecha))
    ws2.cell(row=start_row - 1, column=start_col + col_offset, value="Estacion")
    for i, idx_est in enumerate(orden):
        ws2.cell(row=start_row + i, column=start_col + col_offset, value=estaciones[idx_est])

    # Inventarios
    for inv_idx in range(len(tabla)):
        header_inv = headers_inv[inv_idx % len(headers_inv)]
        col_inv = start_col + col_offset + inv_idx + 1
        ws2.cell(row=start_row - 1, column=col_inv, value=header_inv)
        for i, idx_est in enumerate(orden):
            if idx_est < len(tabla[inv_idx]["inv_final"]):
                ws2.cell(row=start_row + i, column=col_inv, value=tabla[inv_idx]["inv_final"][idx_est])

    # Asimetría y variaciones
    for idx, header in enumerate(headers_asi):
        col_asi_var = start_col + col_offset + len(headers_inv) + 1 + idx
        ws2.cell(row=start_row - 1, column=col_asi_var, value=header)
        for i, idx_est in enumerate(orden):
            if idx_est < len(asimetria_pura):
                valor = asimetria_pura[idx_est] * multipliers[idx]
                ws2.cell(row=start_row + i, column=col_asi_var, value=round(valor, 2))

    # Límites
    col_lim_start = start_col + col_offset + len(headers_inv) + len(headers_asi) + 1
    for lim_idx in range(len(tabla)):
        header_lim = headers_lim[lim_idx % len(headers_lim)]
        col_lim = col_lim_start + lim_idx
        ws2.cell(row=start_row - 1, column=col_lim, value=header_lim)
        for i, idx_est in enumerate(orden):
            if idx_est < len(tabla[lim_idx]["limites"]):
                par = tabla[lim_idx]["limites"][idx_est]
                if isinstance(par, list) and len(par) == 2:
                    valor = f"{round(par[0], 2)}, {round(par[1], 2)}"
                    ws2.cell(row=start_row + i, column=col_lim, value=valor)

    # B inicial
    col_binicial = col_lim + 1
    ws2.cell(row=start_row - 1, column=col_binicial, value="B inicial")
    for i, idx_est in enumerate(orden):
        if idx_est < len(tabla[0]["inv_inicial"]):
            ws2.cell(row=start_row + i, column=col_binicial, value=tabla[0]["inv_inicial"][idx_est])

    # Mmodelo
    col_mmodelo = col_binicial + 1
    ws2.cell(row=start_row - 1, column=col_mmodelo, value="Mmodelo")
    for i, idx_est in enumerate(orden):
        if idx_est < len(tabla[0]["capacidad"]):
            ws2.cell(row=start_row + i, column=col_mmodelo, value=tabla[0]["capacidad"][idx_est])

    # === Agregar columnas V -50, V -25, V 0, V +25, V +50 ===
    v_headers = ["V -50", "V -25", "V 0", "V +25", "V +50"]
    for v_idx, v_header in enumerate(v_headers):
        col_v = col_mmodelo + 1 + v_idx
        ws2.cell(row=start_row - 1, column=col_v, value=v_header)

        # Columna de inventario correspondiente para comparar (inv final -50%, -25%, 0, +25%, +50%)
        col_inv_idx = start_col + col_offset + v_idx + 1  # Corresponde a la posición de headers_inv

        # Aplicar fórmula según filas:
        # primeras 10 filas: si inv final > B inicial → 1, sino 0
        # siguientes 10 filas: si inv final < B inicial → 1, sino 0
        for i, idx_est in enumerate(orden):
            # Obtener valores
            inv_final_val = None
            b_inicial_val = None
            if idx_est < len(tabla[v_idx]["inv_final"]):
                inv_final_val = tabla[v_idx]["inv_final"][idx_est]
            if idx_est < len(tabla[0]["inv_inicial"]):
                b_inicial_val = tabla[0]["inv_inicial"][idx_est]

            # Definir valor según condición
            val = 0
            if i < 10:
                if inv_final_val is not None and b_inicial_val is not None and inv_final_val > b_inicial_val:
                    val = 1
            else:
                if inv_final_val is not None and b_inicial_val is not None and inv_final_val < b_inicial_val:
                    val = 1
            ws2.cell(row=start_row + i, column=col_v, value=val)

        # Ahora sumas al final de la columna V...
        suma_filas_10 = sum(ws2.cell(row=start_row + r, column=col_v).value or 0 for r in range(0,10))
        suma_filas_10_20 = sum(ws2.cell(row=start_row + r, column=col_v).value or 0 for r in range(10,20))
        suma_total_20 = suma_filas_10 + suma_filas_10_20

        # Escribir sumas justo después de las filas de datos
        fila_suma_10 = start_row + 20
        fila_suma_10_20 = start_row + 21
        fila_suma_total = start_row + 22

        ws2.cell(row=fila_suma_10, column=col_v, value=suma_filas_10)
        ws2.cell(row=fila_suma_10_20, column=col_v, value=suma_filas_10_20)
        ws2.cell(row=fila_suma_total, column=col_v, value=suma_total_20)

    tabla_idx += 1

# Guardar archivo
wb.save("resultado_completo.xlsx")


In [21]:
import pandas as pd
from openpyxl import Workbook
import ast

# Leer el Excel principal
df = pd.read_excel("Resultado_optimizaciontodasanalisis.xlsx", sheet_name="Sheet1")
df["Fecha"] = pd.to_datetime(df["Fecha"], errors="coerce")

# Leer el archivo de resumen bikes
bikes_df = pd.read_excel("resumen_bikes.xlsx")

# Estandarizar columnas de fechas en resumen bikes
bikes_df.columns = [pd.to_datetime(str(col), errors='coerce').date() if isinstance(col, str) else col for col in bikes_df.columns]
estaciones_bikes = bikes_df.iloc[:, 0].tolist()
bikes_df.set_index(bikes_df.columns[0], inplace=True)

# Crear libro y hojas
wb = Workbook()
ws1 = wb.active
ws1.title = "Direcciones Organizadas"
ws2 = wb.create_sheet(title="buscarv")

# Parámetros
column_spacing = 7
row_spacing = 4
listas_por_bloque = 5
headers_inv = ["inv final -50%", "inv final -25%", "inv final 0", "inv final +25%", "inv final +50%"]
headers_lim = ["L-50%", "L -25%", "L 0", "L+25%", "L+50%"]
headers_asi = ["S-50", "S-25", "S", "S+25", "S+50"]
multipliers = [0.5, 0.75, 1.0, 1.25, 1.5]

# === Parte 1: Direcciones Organizadas ===
start_row = 2
start_col = 1

for i in range(0, len(df), listas_por_bloque):
    bloque = df["Estaciones"].iloc[i:i+listas_por_bloque]
    for j, item in enumerate(bloque):
        if pd.isna(item):
            continue
        texto = str(item).strip("[]").replace("'", "").replace('"', "")
        direcciones = [x.strip() for x in texto.split(",")]
        col = start_col + j * column_spacing
        ws1.cell(row=start_row - 1, column=col, value="Estacion")
        for k, direccion in enumerate(direcciones):
            ws1.cell(row=start_row + k, column=col, value=direccion)
    max_direcciones = max(len(str(x).strip("[]").split(',')) for x in bloque if pd.notna(x))
    start_row += row_spacing + max_direcciones

# === Parte 2: buscarv ===
start_row = 3
start_col = 1
tabla_idx = 0

for fecha, grupo in df.groupby(df["Fecha"].dt.date):
    col_offset = tabla_idx * (len(headers_inv) + len(headers_lim) + len(headers_asi) + 7 + len(headers_inv))
    primera_fila = grupo.iloc[0]
    estaciones_texto = str(primera_fila["Estaciones"]).strip("[]").replace("'", "").replace('"', "")
    estaciones = [x.strip() for x in estaciones_texto.split(",")]

    try:
        asimetria_pura = ast.literal_eval(str(primera_fila["Asimetriamodelo"]))
    except:
        asimetria_pura = []

    tabla = []
    for idx, (_, fila) in enumerate(grupo.iterrows()):
        try:
            inventario_final = ast.literal_eval(str(fila["Inventario_final"]))
        except:
            inventario_final = []

        try:
            limites = ast.literal_eval(str(fila["Limites"]))
        except:
            limites = []

        tabla.append({
            "inv_final": inventario_final,
            "limites": limites,
            "capacidad": ast.literal_eval(str(fila["Capacidad"])) if pd.notna(fila["Capacidad"]) else [],
            "inv_inicial": ast.literal_eval(str(fila["Inventario_inicial"])) if pd.notna(fila["Inventario_inicial"]) else [],
            "idx": idx,
            "s_base": asimetria_pura
        })

    orden = sorted(range(len(estaciones)), key=lambda i: asimetria_pura[i], reverse=True)

    ws2.cell(row=start_row - 2, column=start_col + col_offset, value=str(fecha))
    ws2.cell(row=start_row - 1, column=start_col + col_offset, value="Estacion")
    for i, idx_est in enumerate(orden):
        ws2.cell(row=start_row + i, column=start_col + col_offset, value=estaciones[idx_est])

    # === Agregar columna B 19:00 ===
    if fecha in bikes_df.columns:
        b_1900_datos_dict = bikes_df[fecha].to_dict()
    else:
        b_1900_datos_dict = {}
    col_b1900 = start_col + col_offset + 1
    ws2.cell(row=start_row - 1, column=col_b1900, value="B 19:00")
    for i, idx_est in enumerate(orden):
        est = estaciones[idx_est]
        valor = b_1900_datos_dict.get(est, None)
        ws2.cell(row=start_row + i, column=col_b1900, value=valor)

    # === Inventarios ===
    for inv_idx in range(len(tabla)):
        header_inv = headers_inv[inv_idx % len(headers_inv)]
        col_inv = start_col + col_offset + inv_idx + 2
        ws2.cell(row=start_row - 1, column=col_inv, value=header_inv)
        for i, idx_est in enumerate(orden):
            if idx_est < len(tabla[inv_idx]["inv_final"]):
                ws2.cell(row=start_row + i, column=col_inv, value=tabla[inv_idx]["inv_final"][idx_est])

    # === Asimetría ===
    for idx, header in enumerate(headers_asi):
        col_asi_var = start_col + col_offset + len(headers_inv) + 2 + idx
        ws2.cell(row=start_row - 1, column=col_asi_var, value=header)
        for i, idx_est in enumerate(orden):
            if idx_est < len(asimetria_pura):
                valor = asimetria_pura[idx_est] * multipliers[idx]
                ws2.cell(row=start_row + i, column=col_asi_var, value=round(valor, 2))

    # === Límites ===
    col_lim_start = start_col + col_offset + len(headers_inv) + len(headers_asi) + 2
    for lim_idx in range(len(tabla)):
        header_lim = headers_lim[lim_idx % len(headers_lim)]
        col_lim = col_lim_start + lim_idx
        ws2.cell(row=start_row - 1, column=col_lim, value=header_lim)
        for i, idx_est in enumerate(orden):
            if idx_est < len(tabla[lim_idx]["limites"]):
                par = tabla[lim_idx]["limites"][idx_est]
                if isinstance(par, list) and len(par) == 2:
                    valor = f"{round(par[0], 2)}, {round(par[1], 2)}"
                    ws2.cell(row=start_row + i, column=col_lim, value=valor)

    col_binicial = col_lim + 1
    ws2.cell(row=start_row - 1, column=col_binicial, value="B inicial")
    for i, idx_est in enumerate(orden):
        if idx_est < len(tabla[0]["inv_inicial"]):
            ws2.cell(row=start_row + i, column=col_binicial, value=tabla[0]["inv_inicial"][idx_est])

    col_mmodelo = col_binicial + 1
    ws2.cell(row=start_row - 1, column=col_mmodelo, value="Mmodelo")
    for i, idx_est in enumerate(orden):
        if idx_est < len(tabla[0]["capacidad"]):
            ws2.cell(row=start_row + i, column=col_mmodelo, value=tabla[0]["capacidad"][idx_est])

    v_headers = ["V -50", "V -25", "V 0", "V +25", "V +50"]
    for v_idx, v_header in enumerate(v_headers):
        col_v = col_mmodelo + 1 + v_idx
        ws2.cell(row=start_row - 1, column=col_v, value=v_header)

        for i, idx_est in enumerate(orden):
            inv_final_val = tabla[v_idx]["inv_final"][idx_est] if idx_est < len(tabla[v_idx]["inv_final"]) else None
            b_inicial_val = tabla[0]["inv_inicial"][idx_est] if idx_est < len(tabla[0]["inv_inicial"]) else None

            val = 0
            if i < 10:
                if inv_final_val is not None and b_inicial_val is not None and inv_final_val > b_inicial_val:
                    val = 1
            else:
                if inv_final_val is not None and b_inicial_val is not None and inv_final_val < b_inicial_val:
                    val = 1
            ws2.cell(row=start_row + i, column=col_v, value=val)

        ws2.cell(row=start_row + 20, column=col_v, value=sum(ws2.cell(row=start_row + r, column=col_v).value or 0 for r in range(0,10)))
        ws2.cell(row=start_row + 21, column=col_v, value=sum(ws2.cell(row=start_row + r, column=col_v).value or 0 for r in range(10,20)))
        ws2.cell(row=start_row + 22, column=col_v, value=sum(ws2.cell(row=start_row + r, column=col_v).value or 0 for r in range(0,20)))

    tabla_idx += 1

# Guardar
wb.save("resultado_completo19.xlsx")


In [29]:
import pandas as pd
from openpyxl import Workbook
import ast

# Leer el Excel principal
df = pd.read_excel("Resultado_optimizaciontodasanalisis.xlsx", sheet_name="Sheet1")
df["Fecha"] = pd.to_datetime(df["Fecha"], errors="coerce")

# Leer el archivo de resumen bikes
bikes_df = pd.read_excel("resumen_bikes.xlsx")

# Estandarizar columnas de fechas en resumen bikes
bikes_df.columns = [pd.to_datetime(str(col), errors='coerce').date() if isinstance(col, str) else col for col in bikes_df.columns]
estaciones_bikes = bikes_df.iloc[:, 0].tolist()
bikes_df.set_index(bikes_df.columns[0], inplace=True)

# Crear libro y hojas
wb = Workbook()
ws1 = wb.active
ws1.title = "Direcciones Organizadas"
ws2 = wb.create_sheet(title="buscarv")

# Parámetros
column_spacing = 7
row_spacing = 4
listas_por_bloque = 5
headers_inv = ["inv final -50%", "inv final -25%", "inv final 0", "inv final +25%", "inv final +50%"]
headers_lim = ["L-50%", "L -25%", "L 0", "L+25%", "L+50%"]
headers_asi = ["S-50", "S-25", "S", "S+25", "S+50"]
multipliers = [0.5, 0.75, 1.0, 1.25, 1.5]

# === Parte 1: Direcciones Organizadas ===
start_row = 2
start_col = 1

for i in range(0, len(df), listas_por_bloque):
    bloque = df["Estaciones"].iloc[i:i+listas_por_bloque]
    for j, item in enumerate(bloque):
        if pd.isna(item):
            continue
        texto = str(item).strip("[]").replace("'", "").replace('"', "")
        direcciones = [x.strip() for x in texto.split(",")]
        col = start_col + j * column_spacing
        ws1.cell(row=start_row - 1, column=col, value="Estacion")
        for k, direccion in enumerate(direcciones):
            ws1.cell(row=start_row + k, column=col, value=direccion)
    max_direcciones = max(len(str(x).strip("[]").split(',')) for x in bloque if pd.notna(x))
    start_row += row_spacing + max_direcciones

# === Parte 2: buscarv ===
start_row = 3
start_col = 1
tabla_idx = 0

for fecha, grupo in df.groupby(df["Fecha"].dt.date):
    col_offset = tabla_idx * (len(headers_inv) + len(headers_lim) + len(headers_asi) + 7 + len(headers_inv))
    primera_fila = grupo.iloc[0]
    estaciones_texto = str(primera_fila["Estaciones"]).strip("[]").replace("'", "").replace('"', "")
    estaciones = [x.strip() for x in estaciones_texto.split(",")]

    try:
        asimetria_pura = ast.literal_eval(str(primera_fila["Asimetriamodelo"]))
    except:
        asimetria_pura = []

    tabla = []
    for idx, (_, fila) in enumerate(grupo.iterrows()):
        try:
            inventario_final = ast.literal_eval(str(fila["Inventario_final"]))
        except:
            inventario_final = []

        try:
            limites = ast.literal_eval(str(fila["Limites"]))
        except:
            limites = []

        tabla.append({
            "inv_final": inventario_final,
            "limites": limites,
            "capacidad": ast.literal_eval(str(fila["Capacidad"])) if pd.notna(fila["Capacidad"]) else [],
            "inv_inicial": ast.literal_eval(str(fila["Inventario_inicial"])) if pd.notna(fila["Inventario_inicial"]) else [],
            "idx": idx,
            "s_base": asimetria_pura
        })

    orden = sorted(range(len(estaciones)), key=lambda i: asimetria_pura[i], reverse=True)

    ws2.cell(row=start_row - 2, column=start_col + col_offset, value=str(fecha))
    ws2.cell(row=start_row - 1, column=start_col + col_offset, value="Estacion")
    for i, idx_est in enumerate(orden):
        ws2.cell(row=start_row + i, column=start_col + col_offset, value=estaciones[idx_est])

    # === Agregar columna B 19:00 ===
    if fecha in bikes_df.columns:
        b_1900_datos_dict = bikes_df[fecha].to_dict()
    else:
        b_1900_datos_dict = {}
    col_b1900 = start_col + col_offset + 1
    ws2.cell(row=start_row - 1, column=col_b1900, value="B 19:00")
    for i, idx_est in enumerate(orden):
        est = estaciones[idx_est]
        valor = b_1900_datos_dict.get(est, None)
        ws2.cell(row=start_row + i, column=col_b1900, value=valor)

    # === Inventarios ===
    for inv_idx in range(len(tabla)):
        header_inv = headers_inv[inv_idx % len(headers_inv)]
        col_inv = start_col + col_offset + inv_idx + 2
        ws2.cell(row=start_row - 1, column=col_inv, value=header_inv)
        for i, idx_est in enumerate(orden):
            if idx_est < len(tabla[inv_idx]["inv_final"]):
                ws2.cell(row=start_row + i, column=col_inv, value=tabla[inv_idx]["inv_final"][idx_est])

    # === Asimetría ===
    for idx, header in enumerate(headers_asi):
        col_asi_var = start_col + col_offset + len(headers_inv) + 2 + idx
        ws2.cell(row=start_row - 1, column=col_asi_var, value=header)
        for i, idx_est in enumerate(orden):
            if idx_est < len(asimetria_pura):
                valor = asimetria_pura[idx_est] * multipliers[idx]
                ws2.cell(row=start_row + i, column=col_asi_var, value=round(valor, 2))

    # === Límites ===
    col_lim_start = start_col + col_offset + len(headers_inv) + len(headers_asi) + 2
    for lim_idx in range(len(tabla)):
        header_lim = headers_lim[lim_idx % len(headers_lim)]
        col_lim = col_lim_start + lim_idx
        ws2.cell(row=start_row - 1, column=col_lim, value=header_lim)
        for i, idx_est in enumerate(orden):
            if idx_est < len(tabla[lim_idx]["limites"]):
                par = tabla[lim_idx]["limites"][idx_est]
                if isinstance(par, list) and len(par) == 2:
                    valor = f"{round(par[0], 2)}, {round(par[1], 2)}"
                    ws2.cell(row=start_row + i, column=col_lim, value=valor)

    col_binicial = col_lim + 1
    ws2.cell(row=start_row - 1, column=col_binicial, value="B inicial")
    for i, idx_est in enumerate(orden):
        if idx_est < len(tabla[0]["inv_inicial"]):
            ws2.cell(row=start_row + i, column=col_binicial, value=tabla[0]["inv_inicial"][idx_est])

    col_mmodelo = col_binicial + 1
    ws2.cell(row=start_row - 1, column=col_mmodelo, value="Mmodelo")
    for i, idx_est in enumerate(orden):
        if idx_est < len(tabla[0]["capacidad"]):
            ws2.cell(row=start_row + i, column=col_mmodelo, value=tabla[0]["capacidad"][idx_est])

    v_headers = ["V -50", "V -25", "V 0", "V +25", "V +50"]
    for v_idx, v_header in enumerate(v_headers):
        col_v = col_mmodelo + 1 + v_idx
        ws2.cell(row=start_row - 1, column=col_v, value=v_header)

        for i, idx_est in enumerate(orden):
            inv_final_val = tabla[v_idx]["inv_final"][idx_est] if idx_est < len(tabla[v_idx]["inv_final"]) else None
            b_1900_val = ws2.cell(row=start_row + i, column=col_b1900).value

            val = 0
            if inv_final_val is not None and b_1900_val is not None:
                try:
                    inv_final_num = float(inv_final_val)
                    b_1900_num = float(b_1900_val)
                    if i < 10:
                        if inv_final_num >= b_1900_num:
                            val = 1
                    else:
                        if inv_final_num <= b_1900_num:
                            val = 1
                except:
                    val = 0
            ws2.cell(row=start_row + i, column=col_v, value=val)

        ws2.cell(row=start_row + 20, column=col_v, value=sum(ws2.cell(row=start_row + r, column=col_v).value or 0 for r in range(0,10)))
        ws2.cell(row=start_row + 21, column=col_v, value=sum(ws2.cell(row=start_row + r, column=col_v).value or 0 for r in range(10,20)))
        ws2.cell(row=start_row + 22, column=col_v, value=sum(ws2.cell(row=start_row + r, column=col_v).value or 0 for r in range(0,20)))

    tabla_idx += 1

# Guardar
wb.save("resultado_completo19final.xlsx")



In [30]:
import pandas as pd
from openpyxl import Workbook
import ast

# Leer el Excel principal
df = pd.read_excel("Resultado_optimizaciontodasanalisis.xlsx", sheet_name="Sheet1")
df["Fecha"] = pd.to_datetime(df["Fecha"], errors="coerce")

# Leer el archivo de resumen bikes
bikes_df = pd.read_excel("resumen_bikes.xlsx")

# Estandarizar columnas de fechas en resumen bikes
bikes_df.columns = [pd.to_datetime(str(col), errors='coerce').date() if isinstance(col, str) else col for col in bikes_df.columns]
estaciones_bikes = bikes_df.iloc[:, 0].tolist()
bikes_df.set_index(bikes_df.columns[0], inplace=True)

# Crear libro y hojas
wb = Workbook()
ws1 = wb.active
ws1.title = "Direcciones Organizadas"
ws2 = wb.create_sheet(title="buscarv")

# Parámetros
column_spacing = 7
row_spacing = 4
listas_por_bloque = 5
headers_inv = ["inv final -50%", "inv final -25%", "inv final 0", "inv final +25%", "inv final +50%"]
headers_lim = ["L-50%", "L -25%", "L 0", "L+25%", "L+50%"]
headers_asi = ["S-50", "S-25", "S", "S+25", "S+50"]
multipliers = [0.5, 0.75, 1.0, 1.25, 1.5]

# === Parte 1: Direcciones Organizadas ===
start_row = 2
start_col = 1

for i in range(0, len(df), listas_por_bloque):
    bloque = df["Estaciones"].iloc[i:i+listas_por_bloque]
    for j, item in enumerate(bloque):
        if pd.isna(item):
            continue
        texto = str(item).strip("[]").replace("'", "").replace('"', "")
        direcciones = [x.strip() for x in texto.split(",")]
        col = start_col + j * column_spacing
        ws1.cell(row=start_row - 1, column=col, value="Estacion")
        for k, direccion in enumerate(direcciones):
            ws1.cell(row=start_row + k, column=col, value=direccion)
    max_direcciones = max(len(str(x).strip("[]").split(',')) for x in bloque if pd.notna(x))
    start_row += row_spacing + max_direcciones

# === Parte 2: buscarv ===
start_row = 3
start_col = 1
tabla_idx = 0

for fecha, grupo in df.groupby(df["Fecha"].dt.date):
    col_offset = tabla_idx * (len(headers_inv) + len(headers_lim) + len(headers_asi) + 7 + len(headers_inv))
    primera_fila = grupo.iloc[0]
    estaciones_texto = str(primera_fila["Estaciones"]).strip("[]").replace("'", "").replace('"', "")
    estaciones = [x.strip() for x in estaciones_texto.split(",")]

    try:
        asimetria_pura = ast.literal_eval(str(primera_fila["Asimetriamodelo"]))
    except:
        asimetria_pura = []

    tabla = []
    for idx, (_, fila) in enumerate(grupo.iterrows()):
        try:
            inventario_final = ast.literal_eval(str(fila["Inventario_final"]))
        except:
            inventario_final = []

        try:
            limites = ast.literal_eval(str(fila["Limites"]))
        except:
            limites = []

        tabla.append({
            "inv_final": inventario_final,
            "limites": limites,
            "capacidad": ast.literal_eval(str(fila["Capacidad"])) if pd.notna(fila["Capacidad"]) else [],
            "inv_inicial": ast.literal_eval(str(fila["Inventario_inicial"])) if pd.notna(fila["Inventario_inicial"]) else [],
            "idx": idx,
            "s_base": asimetria_pura
        })

    orden = sorted(range(len(estaciones)), key=lambda i: asimetria_pura[i], reverse=True)

    ws2.cell(row=start_row - 2, column=start_col + col_offset, value=str(fecha))
    ws2.cell(row=start_row - 1, column=start_col + col_offset, value="Estacion")
    for i, idx_est in enumerate(orden):
        ws2.cell(row=start_row + i, column=start_col + col_offset, value=estaciones[idx_est])

    # === Inventarios ===
    for inv_idx in range(len(tabla)):
        header_inv = headers_inv[inv_idx % len(headers_inv)]
        col_inv = start_col + col_offset + inv_idx + 1
        ws2.cell(row=start_row - 1, column=col_inv, value=header_inv)
        for i, idx_est in enumerate(orden):
            if idx_est < len(tabla[inv_idx]["inv_final"]):
                ws2.cell(row=start_row + i, column=col_inv, value=tabla[inv_idx]["inv_final"][idx_est])

    # === Asimetría ===
    for idx, header in enumerate(headers_asi):
        col_asi_var = start_col + col_offset + len(headers_inv) + 1 + idx
        ws2.cell(row=start_row - 1, column=col_asi_var, value=header)
        for i, idx_est in enumerate(orden):
            if idx_est < len(asimetria_pura):
                valor = asimetria_pura[idx_est] * multipliers[idx]
                ws2.cell(row=start_row + i, column=col_asi_var, value=round(valor, 2))

    # === Límites ===
    col_lim_start = start_col + col_offset + len(headers_inv) + len(headers_asi) + 1
    for lim_idx in range(len(tabla)):
        header_lim = headers_lim[lim_idx % len(headers_lim)]
        col_lim = col_lim_start + lim_idx
        ws2.cell(row=start_row - 1, column=col_lim, value=header_lim)
        for i, idx_est in enumerate(orden):
            if idx_est < len(tabla[lim_idx]["limites"]):
                par = tabla[lim_idx]["limites"][idx_est]
                if isinstance(par, list) and len(par) == 2:
                    valor = f"{round(par[0], 2)}, {round(par[1], 2)}"
                    ws2.cell(row=start_row + i, column=col_lim, value=valor)

    # B inicial
    col_binicial = col_lim + 1
    ws2.cell(row=start_row - 1, column=col_binicial, value="B inicial")
    for i, idx_est in enumerate(orden):
        if idx_est < len(tabla[0]["inv_inicial"]):
            ws2.cell(row=start_row + i, column=col_binicial, value=tabla[0]["inv_inicial"][idx_est])

    # B 19:00 (después de B inicial)
    col_b1900 = col_binicial + 1
    ws2.cell(row=start_row - 1, column=col_b1900, value="B 19:00")
    if fecha in bikes_df.columns:
        b_1900_datos_dict = bikes_df[fecha].to_dict()
    else:
        b_1900_datos_dict = {}
    for i, idx_est in enumerate(orden):
        est = estaciones[idx_est]
        valor = b_1900_datos_dict.get(est, None)
        ws2.cell(row=start_row + i, column=col_b1900, value=valor)

    # Mmodelo
    col_mmodelo = col_b1900 + 1
    ws2.cell(row=start_row - 1, column=col_mmodelo, value="Mmodelo")
    for i, idx_est in enumerate(orden):
        if idx_est < len(tabla[0]["capacidad"]):
            ws2.cell(row=start_row + i, column=col_mmodelo, value=tabla[0]["capacidad"][idx_est])

    # === Fórmulas V ===
    v_headers = ["V -50", "V -25", "V 0", "V +25", "V +50"]
    for v_idx, v_header in enumerate(v_headers):
        col_v = col_mmodelo + 1 + v_idx
        ws2.cell(row=start_row - 1, column=col_v, value=v_header)

        for i, idx_est in enumerate(orden):
            inv_final_val = tabla[v_idx]["inv_final"][idx_est] if idx_est < len(tabla[v_idx]["inv_final"]) else None
            b_1900_val = b_1900_datos_dict.get(estaciones[idx_est], None)
            try:
                if inv_final_val is not None and b_1900_val is not None:
                    b_1900_val = int(b_1900_val)
                    if i < 10:
                        val = 1 if inv_final_val >= b_1900_val else 0
                    else:
                        val = 1 if inv_final_val <= b_1900_val else 0
                else:
                    val = 0
            except:
                val = 0
            ws2.cell(row=start_row + i, column=col_v, value=val)

        ws2.cell(row=start_row + 20, column=col_v, value=sum(ws2.cell(row=start_row + r, column=col_v).value or 0 for r in range(0,10)))
        ws2.cell(row=start_row + 21, column=col_v, value=sum(ws2.cell(row=start_row + r, column=col_v).value or 0 for r in range(10,20)))
        ws2.cell(row=start_row + 22, column=col_v, value=sum(ws2.cell(row=start_row + r, column=col_v).value or 0 for r in range(0,20)))

    tabla_idx += 1

# Guardar archivo final
wb.save("resultado_completo19final2.xlsx")


In [54]:
import pandas as pd
from openpyxl import Workbook
import ast

# Leer el Excel principal
#df = pd.read_excel("Resultado_optimizaciontodasanalisis.xlsx", sheet_name="Sheet1")
df = pd.read_excel("Resultado_optimizaciontodasanalisis.xlsx", sheet_name="Sheet1")
df["Fecha"] = pd.to_datetime(df["Fecha"], errors="coerce")

# Leer el archivo de resumen bikes
bikes_df = pd.read_excel("resumen_bikes.xlsx")

# Estandarizar columnas de fechas en resumen bikes
bikes_df.columns = [pd.to_datetime(str(col), errors='coerce').date() if isinstance(col, str) else col for col in bikes_df.columns]
estaciones_bikes = bikes_df.iloc[:, 0].tolist()
bikes_df.set_index(bikes_df.columns[0], inplace=True)

# Crear libro y hoja "buscarv"
wb = Workbook()
ws2 = wb.active
ws2.title = "buscarv"

# Parámetros
column_spacing = 7
row_spacing = 4
listas_por_bloque = 5
headers_inv = ["inv final -50%", "inv final -25%", "inv final 0", "inv final +25%", "inv final +50%"]
headers_lim = ["L-50%", "L -25%", "L 0", "L+25%", "L+50%"]
headers_asi = ["S-50", "S-25", "S", "S+25", "S+50"]
multipliers = [0.5, 0.75, 1.0, 1.25, 1.5]

# === Procesamiento principal ===
start_row = 3
start_col = 1
tabla_idx = 0

for fecha, grupo in df.groupby(df["Fecha"].dt.date):
    col_offset = tabla_idx * (len(headers_inv) + len(headers_lim) + len(headers_asi) + 7 + len(headers_inv))
    primera_fila = grupo.iloc[0]
    estaciones_texto = str(primera_fila["Estaciones"]).strip("[]").replace("'", "").replace('"', "")
    estaciones = [x.strip() for x in estaciones_texto.split(",")]

    try:
        asimetria_pura = ast.literal_eval(str(primera_fila["Asimetriamodelo"]))
    except:
        asimetria_pura = []

    tabla = []
    for idx, (_, fila) in enumerate(grupo.iterrows()):
        try:
            inventario_final = ast.literal_eval(str(fila["Inventario_final"]))
        except:
            inventario_final = []

        try:
            limites = ast.literal_eval(str(fila["Limites"]))
        except:
            limites = []

        tabla.append({
            "inv_final": inventario_final,
            "limites": limites,
            "capacidad": ast.literal_eval(str(fila["Capacidad"])) if pd.notna(fila["Capacidad"]) else [],
            "inv_inicial": ast.literal_eval(str(fila["Inventario_inicial"])) if pd.notna(fila["Inventario_inicial"]) else [],
            "idx": idx,
            "s_base": asimetria_pura
        })

    orden = sorted(range(len(estaciones)), key=lambda i: asimetria_pura[i], reverse=True)

    ws2.cell(row=start_row - 2, column=start_col + col_offset, value=str(fecha))
    ws2.cell(row=start_row - 1, column=start_col + col_offset, value="Estacion")
    for i, idx_est in enumerate(orden):
        ws2.cell(row=start_row + i, column=start_col + col_offset, value=estaciones[idx_est])

    # === Inventarios ===
    for inv_idx in range(len(tabla)):
        header_inv = headers_inv[inv_idx % len(headers_inv)]
        col_inv = start_col + col_offset + inv_idx + 1
        ws2.cell(row=start_row - 1, column=col_inv, value=header_inv)
        for i, idx_est in enumerate(orden):
            if idx_est < len(tabla[inv_idx]["inv_final"]):
                ws2.cell(row=start_row + i, column=col_inv, value=tabla[inv_idx]["inv_final"][idx_est])

    # === Asimetría ===
    for idx, header in enumerate(headers_asi):
        col_asi_var = start_col + col_offset + len(headers_inv) + 1 + idx
        ws2.cell(row=start_row - 1, column=col_asi_var, value=header)
        for i, idx_est in enumerate(orden):
            if idx_est < len(asimetria_pura):
                valor = asimetria_pura[idx_est] * multipliers[idx]
                ws2.cell(row=start_row + i, column=col_asi_var, value=round(valor, 2))

    # === Límites ===
    col_lim_start = start_col + col_offset + len(headers_inv) + len(headers_asi) + 1
    for lim_idx in range(len(tabla)):
        header_lim = headers_lim[lim_idx % len(headers_lim)]
        col_lim = col_lim_start + lim_idx
        ws2.cell(row=start_row - 1, column=col_lim, value=header_lim)
        for i, idx_est in enumerate(orden):
            if idx_est < len(tabla[lim_idx]["limites"]):
                par = tabla[lim_idx]["limites"][idx_est]
                if isinstance(par, list) and len(par) == 2:
                    valor = f"{round(par[0], 2)}, {round(par[1], 2)}"
                    ws2.cell(row=start_row + i, column=col_lim, value=valor)

    # === B inicial ===
    col_binicial = col_lim + 1
    ws2.cell(row=start_row - 1, column=col_binicial, value="B inicial")
    for i, idx_est in enumerate(orden):
        if idx_est < len(tabla[0]["inv_inicial"]):
            ws2.cell(row=start_row + i, column=col_binicial, value=tabla[0]["inv_inicial"][idx_est])

    # === B 19:00 ===
    col_b1900 = col_binicial + 1
    ws2.cell(row=start_row - 1, column=col_b1900, value="B 19:00")
    if fecha in bikes_df.columns:
        b_1900_datos_dict = bikes_df[fecha].to_dict()
    else:
        b_1900_datos_dict = {}
    for i, idx_est in enumerate(orden):
        est = estaciones[idx_est]
        valor = b_1900_datos_dict.get(est, None)
        ws2.cell(row=start_row + i, column=col_b1900, value=valor)

    # === Mmodelo ===
    col_mmodelo = col_b1900 + 1
    ws2.cell(row=start_row - 1, column=col_mmodelo, value="Mmodelo")
    for i, idx_est in enumerate(orden):
        if idx_est < len(tabla[0]["capacidad"]):
            ws2.cell(row=start_row + i, column=col_mmodelo, value=tabla[0]["capacidad"][idx_est])

    # === V columnas ===
    v_headers = ["V -50", "V -25", "V 0", "V +25", "V +50"]
    for v_idx, v_header in enumerate(v_headers):
        col_v = col_mmodelo + 1 + v_idx
        ws2.cell(row=start_row - 1, column=col_v, value=v_header)

        for i, idx_est in enumerate(orden):
            inv_final_val = tabla[v_idx]["inv_final"][idx_est] if idx_est < len(tabla[v_idx]["inv_final"]) else None
            b_1900_val = ws2.cell(row=start_row + i, column=col_b1900).value

            try:
                b_1900_val = float(b_1900_val)
            except (TypeError, ValueError):
                b_1900_val = None

            val = 0
            if inv_final_val is not None and b_1900_val is not None:
                if i < 10:
                    val = 1 if inv_final_val >= b_1900_val else 0
                else:
                    val = 1 if inv_final_val <= b_1900_val else 0
            ws2.cell(row=start_row + i, column=col_v, value=val)

        ws2.cell(row=start_row + 20, column=col_v,
                 value=sum(ws2.cell(row=start_row + r, column=col_v).value or 0 for r in range(0, 10)))
        ws2.cell(row=start_row + 21, column=col_v,
                 value=sum(ws2.cell(row=start_row + r, column=col_v).value or 0 for r in range(10, 20)))
        ws2.cell(row=start_row + 22, column=col_v,
                 value=sum(ws2.cell(row=start_row + r, column=col_v).value or 0 for r in range(0, 20)))

    tabla_idx += 1

# Guardar archivo
wb.save("Resultado_completoaver.xlsx")


In [2]:
import pandas as pd
from openpyxl import Workbook
import ast
import statistics

# Leer datos
df = pd.read_excel("Resultado_optimizaciontembicifinal.xlsx", sheet_name="Sheet1")
df["Fecha"] = pd.to_datetime(df["Fecha"], errors="coerce")
bikes_df = pd.read_excel("resumen_bikescasotembici.xlsx")
bikes_df.columns = [pd.to_datetime(str(col), errors='coerce').date() if isinstance(col, str) else col for col in bikes_df.columns]
estaciones_bikes = bikes_df.iloc[:, 0].tolist()
bikes_df.set_index(bikes_df.columns[0], inplace=True)

# Libro
wb = Workbook()
ws2 = wb.active
ws2.title = "buscarv"

# Parámetros
start_row = 3
start_col = 1
tabla_idx = 0
headers_inv = ["inv final -50%", "inv final -25%", "inv final 0", "inv final +25%", "inv final +50%"]
headers_lim = ["L-50%", "L -25%", "L 0", "L+25%", "L+50%"]
headers_asi = ["S-50", "S-25", "S", "S+25", "S+50"]
v_headers = ["V -50", "V -25", "V 0", "V +25", "V +50"]
multipliers = [0.5, 0.75, 1.0, 1.25, 1.5]

for fecha, grupo in df.groupby(df["Fecha"].dt.date):
    col_offset = tabla_idx * (len(headers_inv) + len(headers_lim) + len(headers_asi) + 7 + len(headers_inv))
    primera_fila = grupo.iloc[0]
    estaciones_texto = str(primera_fila["Estaciones"]).strip("[]").replace("'", "").replace('"', "")
    estaciones = [x.strip() for x in estaciones_texto.split(",")]

    try:
        asimetria_pura = ast.literal_eval(str(primera_fila["Asimetriamodelo"]))
    except:
        asimetria_pura = []

    tabla = []
    for idx, (_, fila) in enumerate(grupo.iterrows()):
        tabla.append({
            "inv_final": ast.literal_eval(str(fila["Inventario_final"])) if pd.notna(fila["Inventario_final"]) else [],
            "limites": ast.literal_eval(str(fila["Limites"])) if pd.notna(fila["Limites"]) else [],
            "capacidad": ast.literal_eval(str(fila["Capacidad"])) if pd.notna(fila["Capacidad"]) else [],
            "inv_inicial": ast.literal_eval(str(fila["Inventario_inicial"])) if pd.notna(fila["Inventario_inicial"]) else [],
            "idx": idx,
            "s_base": asimetria_pura
        })

    orden = sorted(range(len(estaciones)), key=lambda i: asimetria_pura[i], reverse=True)

    ws2.cell(row=start_row - 2, column=start_col + col_offset, value=str(fecha))
    ws2.cell(row=start_row - 1, column=start_col + col_offset, value="Estacion")
    for i, idx_est in enumerate(orden):
        ws2.cell(row=start_row + i, column=start_col + col_offset, value=estaciones[idx_est])

    for inv_idx in range(len(tabla)):
        col_inv = start_col + col_offset + inv_idx + 1
        ws2.cell(row=start_row - 1, column=col_inv, value=headers_inv[inv_idx % len(headers_inv)])
        for i, idx_est in enumerate(orden):
            if idx_est < len(tabla[inv_idx]["inv_final"]):
                ws2.cell(row=start_row + i, column=col_inv, value=tabla[inv_idx]["inv_final"][idx_est])

    for idx, header in enumerate(headers_asi):
        col_asi_var = start_col + col_offset + len(headers_inv) + 1 + idx
        ws2.cell(row=start_row - 1, column=col_asi_var, value=header)
        for i, idx_est in enumerate(orden):
            if idx_est < len(asimetria_pura):
                valor = asimetria_pura[idx_est] * multipliers[idx]
                ws2.cell(row=start_row + i, column=col_asi_var, value=round(valor, 2))

    col_lim_start = start_col + col_offset + len(headers_inv) + len(headers_asi) + 1
    for lim_idx in range(len(tabla)):
        col_lim = col_lim_start + lim_idx
        ws2.cell(row=start_row - 1, column=col_lim, value=headers_lim[lim_idx % len(headers_lim)])
        for i, idx_est in enumerate(orden):
            if idx_est < len(tabla[lim_idx]["limites"]):
                par = tabla[lim_idx]["limites"][idx_est]
                if isinstance(par, list) and len(par) == 2:
                    valor = f"{round(par[0], 2)}, {round(par[1], 2)}"
                    ws2.cell(row=start_row + i, column=col_lim, value=valor)

    col_binicial = col_lim + 1
    ws2.cell(row=start_row - 1, column=col_binicial, value="B inicial")
    for i, idx_est in enumerate(orden):
        if idx_est < len(tabla[0]["inv_inicial"]):
            ws2.cell(row=start_row + i, column=col_binicial, value=tabla[0]["inv_inicial"][idx_est])

    col_b1900 = col_binicial + 1
    ws2.cell(row=start_row - 1, column=col_b1900, value="B 19:00")
    b_1900_datos_dict = bikes_df[fecha].to_dict() if fecha in bikes_df.columns else {}
    for i, idx_est in enumerate(orden):
        est = estaciones[idx_est]
        valor = b_1900_datos_dict.get(est, None)
        ws2.cell(row=start_row + i, column=col_b1900, value=valor)

    col_mmodelo = col_b1900 + 1
    ws2.cell(row=start_row - 1, column=col_mmodelo, value="Mmodelo")
    for i, idx_est in enumerate(orden):
        if idx_est < len(tabla[0]["capacidad"]):
            ws2.cell(row=start_row + i, column=col_mmodelo, value=tabla[0]["capacidad"][idx_est])

    # === Columnas V ===
    suma_totales = []
    for v_idx, v_header in enumerate(v_headers):
        col_v = col_mmodelo + 1 + v_idx
        ws2.cell(row=start_row - 1, column=col_v, value=v_header)

        for i, idx_est in enumerate(orden):
            inv_final_val = tabla[v_idx]["inv_final"][idx_est] if idx_est < len(tabla[v_idx]["inv_final"]) else None
            b_1900_val = ws2.cell(row=start_row + i, column=col_b1900).value
            try:
                b_1900_val = float(b_1900_val)
            except (TypeError, ValueError):
                b_1900_val = None
            val = 0
            if inv_final_val is not None and b_1900_val is not None:
                if i < 10:
                    val = 1 if inv_final_val >= b_1900_val else 0
                else:
                    val = 1 if inv_final_val <= b_1900_val else 0
            ws2.cell(row=start_row + i, column=col_v, value=val)

        suma_1_10 = sum(ws2.cell(row=start_row + r, column=col_v).value or 0 for r in range(0, 10))
        suma_11_20 = sum(ws2.cell(row=start_row + r, column=col_v).value or 0 for r in range(10, 20))
        total_20 = suma_1_10 + suma_11_20
        suma_totales.append(total_20)

        ws2.cell(row=start_row + 20, column=col_v, value=suma_1_10)
        ws2.cell(row=start_row + 21, column=col_v, value=suma_11_20)
        ws2.cell(row=start_row + 22, column=col_v, value=total_20)
        variacion_pct = round((total_20 / 20) * 100, 2)
        ws2.cell(row=start_row + 23, column=col_v, value=f"{variacion_pct}%")

    # ✅ Promedio y Desviación estándar en columna "V -50"
    promedio_total = round(sum(suma_totales) / len(suma_totales), 2)
    std_dev = round(statistics.stdev(suma_totales), 4) if len(suma_totales) >= 2 else 0
    col_v50 = col_mmodelo + 1
    ws2.cell(row=start_row + 24, column=col_v50 - 1, value="Promedio")
    ws2.cell(row=start_row + 24, column=col_v50, value=promedio_total)
    ws2.cell(row=start_row + 25, column=col_v50 - 1, value="Desviacion estandar")
    ws2.cell(row=start_row + 25, column=col_v50, value=std_dev)

    tabla_idx += 1

wb.save("Resultado_completoapromediocasotembici.xlsx")

In [5]:
#con promedio de balances correctos porcentaje
import pandas as pd
from openpyxl import Workbook
import ast
import statistics

# Leer datos
df = pd.read_excel("Resultado_optimizaciontembicifinal.xlsx", sheet_name="Sheet1")
df["Fecha"] = pd.to_datetime(df["Fecha"], errors="coerce")
bikes_df = pd.read_excel("resumen_bikescasotembici.xlsx")
bikes_df.columns = [pd.to_datetime(str(col), errors='coerce').date() if isinstance(col, str) else col for col in bikes_df.columns]
estaciones_bikes = bikes_df.iloc[:, 0].tolist()
bikes_df.set_index(bikes_df.columns[0], inplace=True)

# Libro
wb = Workbook()
ws2 = wb.active
ws2.title = "buscarv"

# Parámetros
start_row = 3
start_col = 1
tabla_idx = 0
headers_inv = ["inv final -50%", "inv final -25%", "inv final 0", "inv final +25%", "inv final +50%"]
headers_lim = ["L-50%", "L -25%", "L 0", "L+25%", "L+50%"]
headers_asi = ["S-50", "S-25", "S", "S+25", "S+50"]
v_headers = ["V -50", "V -25", "V 0", "V +25", "V +50"]
multipliers = [0.5, 0.75, 1.0, 1.25, 1.5]

# Totales para calcular promedios globales
todos_los_promedios = []
todas_las_desviaciones = []
todos_los_porcentajes = []

for fecha, grupo in df.groupby(df["Fecha"].dt.date):
    col_offset = tabla_idx * (len(headers_inv) + len(headers_lim) + len(headers_asi) + 7 + len(headers_inv))
    primera_fila = grupo.iloc[0]
    estaciones_texto = str(primera_fila["Estaciones"]).strip("[]").replace("'", "").replace('"', "")
    estaciones = [x.strip() for x in estaciones_texto.split(",")]

    try:
        asimetria_pura = ast.literal_eval(str(primera_fila["Asimetriamodelo"]))
    except:
        asimetria_pura = []

    tabla = []
    for idx, (_, fila) in enumerate(grupo.iterrows()):
        tabla.append({
            "inv_final": ast.literal_eval(str(fila["Inventario_final"])) if pd.notna(fila["Inventario_final"]) else [],
            "limites": ast.literal_eval(str(fila["Limites"])) if pd.notna(fila["Limites"]) else [],
            "capacidad": ast.literal_eval(str(fila["Capacidad"])) if pd.notna(fila["Capacidad"]) else [],
            "inv_inicial": ast.literal_eval(str(fila["Inventario_inicial"])) if pd.notna(fila["Inventario_inicial"]) else [],
            "idx": idx,
            "s_base": asimetria_pura
        })

    orden = sorted(range(len(estaciones)), key=lambda i: asimetria_pura[i], reverse=True)

    ws2.cell(row=start_row - 2, column=start_col + col_offset, value=str(fecha))
    ws2.cell(row=start_row - 1, column=start_col + col_offset, value="Estacion")
    for i, idx_est in enumerate(orden):
        ws2.cell(row=start_row + i, column=start_col + col_offset, value=estaciones[idx_est])

    for inv_idx in range(len(tabla)):
        col_inv = start_col + col_offset + inv_idx + 1
        ws2.cell(row=start_row - 1, column=col_inv, value=headers_inv[inv_idx % len(headers_inv)])
        for i, idx_est in enumerate(orden):
            if idx_est < len(tabla[inv_idx]["inv_final"]):
                ws2.cell(row=start_row + i, column=col_inv, value=tabla[inv_idx]["inv_final"][idx_est])

    for idx, header in enumerate(headers_asi):
        col_asi_var = start_col + col_offset + len(headers_inv) + 1 + idx
        ws2.cell(row=start_row - 1, column=col_asi_var, value=header)
        for i, idx_est in enumerate(orden):
            if idx_est < len(asimetria_pura):
                valor = asimetria_pura[idx_est] * multipliers[idx]
                ws2.cell(row=start_row + i, column=col_asi_var, value=round(valor, 2))

    col_lim_start = start_col + col_offset + len(headers_inv) + len(headers_asi) + 1
    for lim_idx in range(len(tabla)):
        col_lim = col_lim_start + lim_idx
        ws2.cell(row=start_row - 1, column=col_lim, value=headers_lim[lim_idx % len(headers_lim)])
        for i, idx_est in enumerate(orden):
            if idx_est < len(tabla[lim_idx]["limites"]):
                par = tabla[lim_idx]["limites"][idx_est]
                if isinstance(par, list) and len(par) == 2:
                    valor = f"{round(par[0], 2)}, {round(par[1], 2)}"
                    ws2.cell(row=start_row + i, column=col_lim, value=valor)

    col_binicial = col_lim + 1
    ws2.cell(row=start_row - 1, column=col_binicial, value="B inicial")
    for i, idx_est in enumerate(orden):
        if idx_est < len(tabla[0]["inv_inicial"]):
            ws2.cell(row=start_row + i, column=col_binicial, value=tabla[0]["inv_inicial"][idx_est])

    col_b1900 = col_binicial + 1
    ws2.cell(row=start_row - 1, column=col_b1900, value="B 19:00")
    b_1900_datos_dict = bikes_df[fecha].to_dict() if fecha in bikes_df.columns else {}
    for i, idx_est in enumerate(orden):
        est = estaciones[idx_est]
        valor = b_1900_datos_dict.get(est, None)
        ws2.cell(row=start_row + i, column=col_b1900, value=valor)

    col_mmodelo = col_b1900 + 1
    ws2.cell(row=start_row - 1, column=col_mmodelo, value="Mmodelo")
    for i, idx_est in enumerate(orden):
        if idx_est < len(tabla[0]["capacidad"]):
            ws2.cell(row=start_row + i, column=col_mmodelo, value=tabla[0]["capacidad"][idx_est])

    # === Columnas V ===
    suma_totales = []
    porcentajes_var = []
    for v_idx, v_header in enumerate(v_headers):
        col_v = col_mmodelo + 1 + v_idx
        ws2.cell(row=start_row - 1, column=col_v, value=v_header)

        for i, idx_est in enumerate(orden):
            inv_final_val = tabla[v_idx]["inv_final"][idx_est] if idx_est < len(tabla[v_idx]["inv_final"]) else None
            b_1900_val = ws2.cell(row=start_row + i, column=col_b1900).value
            try:
                b_1900_val = float(b_1900_val)
            except (TypeError, ValueError):
                b_1900_val = None
            val = 0
            if inv_final_val is not None and b_1900_val is not None:
                if i < 10:
                    val = 1 if inv_final_val >= b_1900_val else 0
                else:
                    val = 1 if inv_final_val <= b_1900_val else 0
            ws2.cell(row=start_row + i, column=col_v, value=val)

        suma_1_10 = sum(ws2.cell(row=start_row + r, column=col_v).value or 0 for r in range(0, 10))
        suma_11_20 = sum(ws2.cell(row=start_row + r, column=col_v).value or 0 for r in range(10, 20))
        total_20 = suma_1_10 + suma_11_20
        suma_totales.append(total_20)

        ws2.cell(row=start_row + 20, column=col_v, value=suma_1_10)
        ws2.cell(row=start_row + 21, column=col_v, value=suma_11_20)
        ws2.cell(row=start_row + 22, column=col_v, value=total_20)
        porcentaje = round((total_20 / 20) * 100)
        cell_porcentaje = ws2.cell(row=start_row + 23, column=col_v, value=porcentaje / 100)
        cell_porcentaje.number_format = '0%'
        porcentajes_var.append(porcentaje)

    # Promedio, Desviación estándar y Promedio %
    promedio_total = round(sum(suma_totales) / len(suma_totales), 2)
    std_dev = round(statistics.stdev(suma_totales), 4) if len(suma_totales) >= 2 else 0
    promedio_porcentaje = round(sum(porcentajes_var) / len(porcentajes_var))

    col_v50 = col_mmodelo + 1
    ws2.cell(row=start_row + 24, column=col_v50 - 1, value="Promedio")
    ws2.cell(row=start_row + 24, column=col_v50, value=promedio_total)
    ws2.cell(row=start_row + 25, column=col_v50 - 1, value="Desviacion estandar")
    ws2.cell(row=start_row + 25, column=col_v50, value=std_dev)
    ws2.cell(row=start_row + 26, column=col_v50 - 1, value="Promedio %")
    cell_promedio_porcentaje = ws2.cell(row=start_row + 26, column=col_v50, value=promedio_porcentaje / 100)
    cell_promedio_porcentaje.number_format = '0%'

    todos_los_promedios.append(promedio_total)
    todas_las_desviaciones.append(std_dev)
    todos_los_porcentajes.append(promedio_porcentaje)

    tabla_idx += 1

# Promedios globales al final de la primera tabla
final_row = start_row + 30
ws2.cell(row=final_row, column=1, value="Promedio general conteo")
ws2.cell(row=final_row, column=2, value=round(sum(todos_los_promedios) / len(todos_los_promedios), 2))
ws2.cell(row=final_row + 1, column=1, value="Desviacion estandar promedio")
ws2.cell(row=final_row + 1, column=2, value=round(sum(todas_las_desviaciones) / len(todas_las_desviaciones), 4))
ws2.cell(row=final_row + 2, column=1, value="Promedio general %")
prom_final = int(round(sum(todos_los_porcentajes) / len(todos_los_porcentajes)))
cell_prom_final = ws2.cell(row=final_row + 2, column=2, value=prom_final / 100)
cell_prom_final.number_format = '0%'

# Guardar archivo
wb.save("Resultado_completoapromediocasotembicifinalmente.xlsx")


In [8]:
import pandas as pd
from openpyxl import Workbook
import ast
import statistics

# Leer datos
df = pd.read_excel("Resultado_optimizaciontodasanalisisfinal.xlsx", sheet_name="Sheet1")
df["Fecha"] = pd.to_datetime(df["Fecha"], errors="coerce")
bikes_df = pd.read_excel("resumen_bikestodas.xlsx")
bikes_df.columns = [pd.to_datetime(str(col), errors='coerce').date() if isinstance(col, str) else col for col in bikes_df.columns]
estaciones_bikes = bikes_df.iloc[:, 0].tolist()
bikes_df.set_index(bikes_df.columns[0], inplace=True)

# Libro
wb = Workbook()
ws2 = wb.active
ws2.title = "buscarv"

# Parámetros
start_row = 3
start_col = 1
tabla_idx = 0
headers_inv = ["inv final -50%", "inv final -25%", "inv final 0", "inv final +25%", "inv final +50%"]
headers_lim = ["L-50%", "L -25%", "L 0", "L+25%", "L+50%"]
headers_asi = ["S-50", "S-25", "S", "S+25", "S+50"]
v_headers = ["V -50", "V -25", "V 0", "V +25", "V +50"]
multipliers = [0.5, 0.75, 1.0, 1.25, 1.5]

# Totales
todos_los_promedios = []
todas_las_desviaciones = []
todos_los_porcentajes = []
desviaciones_porcentaje = []

for fecha, grupo in df.groupby(df["Fecha"].dt.date):
    col_offset = tabla_idx * (len(headers_inv) + len(headers_lim) + len(headers_asi) + 7 + len(headers_inv))
    primera_fila = grupo.iloc[0]
    estaciones_texto = str(primera_fila["Estaciones"]).strip("[]").replace("'", "").replace('"', "")
    estaciones = [x.strip() for x in estaciones_texto.split(",")]

    try:
        asimetria_pura = ast.literal_eval(str(primera_fila["Asimetriamodelo"]))
    except:
        asimetria_pura = []

    tabla = []
    for idx, (_, fila) in enumerate(grupo.iterrows()):
        tabla.append({
            "inv_final": ast.literal_eval(str(fila["Inventario_final"])) if pd.notna(fila["Inventario_final"]) else [],
            "limites": ast.literal_eval(str(fila["Limites"])) if pd.notna(fila["Limites"]) else [],
            "capacidad": ast.literal_eval(str(fila["Capacidad"])) if pd.notna(fila["Capacidad"]) else [],
            "inv_inicial": ast.literal_eval(str(fila["Inventario_inicial"])) if pd.notna(fila["Inventario_inicial"]) else [],
            "idx": idx,
            "s_base": asimetria_pura
        })

    orden = sorted(range(len(estaciones)), key=lambda i: asimetria_pura[i], reverse=True)

    ws2.cell(row=start_row - 2, column=start_col + col_offset, value=str(fecha))
    ws2.cell(row=start_row - 1, column=start_col + col_offset, value="Estacion")
    for i, idx_est in enumerate(orden):
        ws2.cell(row=start_row + i, column=start_col + col_offset, value=estaciones[idx_est])

    for inv_idx in range(len(tabla)):
        col_inv = start_col + col_offset + inv_idx + 1
        ws2.cell(row=start_row - 1, column=col_inv, value=headers_inv[inv_idx % len(headers_inv)])
        for i, idx_est in enumerate(orden):
            if idx_est < len(tabla[inv_idx]["inv_final"]):
                ws2.cell(row=start_row + i, column=col_inv, value=tabla[inv_idx]["inv_final"][idx_est])

    for idx, header in enumerate(headers_asi):
        col_asi_var = start_col + col_offset + len(headers_inv) + 1 + idx
        ws2.cell(row=start_row - 1, column=col_asi_var, value=header)
        for i, idx_est in enumerate(orden):
            if idx_est < len(asimetria_pura):
                valor = asimetria_pura[idx_est] * multipliers[idx]
                ws2.cell(row=start_row + i, column=col_asi_var, value=round(valor, 2))

    col_lim_start = start_col + col_offset + len(headers_inv) + len(headers_asi) + 1
    for lim_idx in range(len(tabla)):
        col_lim = col_lim_start + lim_idx
        ws2.cell(row=start_row - 1, column=col_lim, value=headers_lim[lim_idx % len(headers_lim)])
        for i, idx_est in enumerate(orden):
            if idx_est < len(tabla[lim_idx]["limites"]):
                par = tabla[lim_idx]["limites"][idx_est]
                if isinstance(par, list) and len(par) == 2:
                    valor = f"{round(par[0], 2)}, {round(par[1], 2)}"
                    ws2.cell(row=start_row + i, column=col_lim, value=valor)

    col_binicial = col_lim + 1
    ws2.cell(row=start_row - 1, column=col_binicial, value="B inicial")
    for i, idx_est in enumerate(orden):
        if idx_est < len(tabla[0]["inv_inicial"]):
            ws2.cell(row=start_row + i, column=col_binicial, value=tabla[0]["inv_inicial"][idx_est])

    col_b1900 = col_binicial + 1
    ws2.cell(row=start_row - 1, column=col_b1900, value="B 19:00")
    b_1900_datos_dict = bikes_df[fecha].to_dict() if fecha in bikes_df.columns else {}
    for i, idx_est in enumerate(orden):
        est = estaciones[idx_est]
        valor = b_1900_datos_dict.get(est, None)
        ws2.cell(row=start_row + i, column=col_b1900, value=valor)

    col_mmodelo = col_b1900 + 1
    ws2.cell(row=start_row - 1, column=col_mmodelo, value="Mmodelo")
    for i, idx_est in enumerate(orden):
        if idx_est < len(tabla[0]["capacidad"]):
            ws2.cell(row=start_row + i, column=col_mmodelo, value=tabla[0]["capacidad"][idx_est])

    # === Columnas V ===
    suma_totales = []
    porcentajes_var = []
    for v_idx, v_header in enumerate(v_headers):
        col_v = col_mmodelo + 1 + v_idx
        ws2.cell(row=start_row - 1, column=col_v, value=v_header)

        for i, idx_est in enumerate(orden):
            inv_final_val = tabla[v_idx]["inv_final"][idx_est] if idx_est < len(tabla[v_idx]["inv_final"]) else None
            b_1900_val = ws2.cell(row=start_row + i, column=col_b1900).value
            try:
                b_1900_val = float(b_1900_val)
            except (TypeError, ValueError):
                b_1900_val = None
            val = 0
            if inv_final_val is not None and b_1900_val is not None:
                if i < 10:
                    val = 1 if inv_final_val >= b_1900_val else 0
                else:
                    val = 1 if inv_final_val <= b_1900_val else 0
            ws2.cell(row=start_row + i, column=col_v, value=val)

        suma_1_10 = sum(ws2.cell(row=start_row + r, column=col_v).value or 0 for r in range(0, 10))
        suma_11_20 = sum(ws2.cell(row=start_row + r, column=col_v).value or 0 for r in range(10, 20))
        total_20 = suma_1_10 + suma_11_20
        suma_totales.append(total_20)

        ws2.cell(row=start_row + 20, column=col_v, value=suma_1_10)
        ws2.cell(row=start_row + 21, column=col_v, value=suma_11_20)
        ws2.cell(row=start_row + 22, column=col_v, value=total_20)
        porcentaje = round((total_20 / 20) * 100)
        cell_porcentaje = ws2.cell(row=start_row + 23, column=col_v, value=porcentaje / 100)
        cell_porcentaje.number_format = '0%'
        porcentajes_var.append(porcentaje)

    # === Métricas de resumen por fecha ===
    promedio_total = round(sum(suma_totales) / len(suma_totales), 2)
    std_dev = round(statistics.stdev(suma_totales), 4) if len(suma_totales) >= 2 else 0
    promedio_porcentaje = round(sum(porcentajes_var) / len(porcentajes_var))
    std_dev_porcentaje = round(statistics.stdev(porcentajes_var), 2) if len(porcentajes_var) >= 2 else 0

    col_v50 = col_mmodelo + 1
    ws2.cell(row=start_row + 24, column=col_v50 - 1, value="Promedio")
    ws2.cell(row=start_row + 24, column=col_v50, value=promedio_total)
    ws2.cell(row=start_row + 25, column=col_v50 - 1, value="Desviacion estandar")
    ws2.cell(row=start_row + 25, column=col_v50, value=std_dev)
    ws2.cell(row=start_row + 26, column=col_v50 - 1, value="Promedio %")
    cell_promedio_porcentaje = ws2.cell(row=start_row + 26, column=col_v50, value=promedio_porcentaje / 100)
    cell_promedio_porcentaje.number_format = '0%'
    ws2.cell(row=start_row + 27, column=col_v50 - 1, value="Desv. estándar % variacion")
    ws2.cell(row=start_row + 27, column=col_v50, value=std_dev_porcentaje)

    todos_los_promedios.append(promedio_total)
    todas_las_desviaciones.append(std_dev)
    todos_los_porcentajes.append(promedio_porcentaje)
    desviaciones_porcentaje.append(std_dev_porcentaje)

    tabla_idx += 1

# === Promedios globales al final ===
final_row = start_row + 30
ws2.cell(row=final_row, column=1, value="Promedio general conteo")
ws2.cell(row=final_row, column=2, value=round(sum(todos_los_promedios) / len(todos_los_promedios), 2))
ws2.cell(row=final_row + 1, column=1, value="Desviacion estandar promedio")
ws2.cell(row=final_row + 1, column=2, value=round(sum(todas_las_desviaciones) / len(todas_las_desviaciones), 4))
ws2.cell(row=final_row + 2, column=1, value="Promedio general %")
prom_final = int(round(sum(todos_los_porcentajes) / len(todos_los_porcentajes)))
cell_prom_final = ws2.cell(row=final_row + 2, column=2, value=prom_final / 100)
cell_prom_final.number_format = '0%'
ws2.cell(row=final_row + 3, column=1, value="Promedio desv. estándar % variación")
ws2.cell(row=final_row + 3, column=2, value=round(sum(desviaciones_porcentaje) / len(desviaciones_porcentaje), 2))

# Guardar archivo
wb.save("Resultado_completoapromediocasotodasdesv.xlsx")


In [3]:
#intentar modificar a mano
import pandas as pd
from openpyxl import Workbook
import ast
import statistics

# Leer datos
df = pd.read_excel("Resultado_optimizaciontembicifinal.xlsx", sheet_name="Sheet1")
df["Fecha"] = pd.to_datetime(df["Fecha"], errors="coerce")
bikes_df = pd.read_excel("resumen_bikescasotembici.xlsx")
bikes_df.columns = [pd.to_datetime(str(col), errors='coerce').date() if isinstance(col, str) else col for col in bikes_df.columns]
estaciones_bikes = bikes_df.iloc[:, 0].tolist()
bikes_df.set_index(bikes_df.columns[0], inplace=True)

# Libro
wb = Workbook()
ws2 = wb.active
ws2.title = "buscarv"

# Parámetros
start_row = 3
start_col = 1
tabla_idx = 0
promedios_suma_1_10 = []
promedios_suma_11_20 = []
headers_inv = ["inv final -50%", "inv final -25%", "inv final 0", "inv final +25%", "inv final +50%"]
headers_lim = ["L-50%", "L -25%", "L 0", "L+25%", "L+50%"]
headers_asi = ["S-50", "S-25", "S", "S+25", "S+50"]
v_headers = ["V -50", "V -25", "V 0", "V +25", "V +50"]
multipliers = [0.5, 0.75, 1.0, 1.25, 1.5]

# Totales
todos_los_promedios = []
todas_las_desviaciones = []
todos_los_porcentajes = []
desviaciones_porcentaje = []

for fecha, grupo in df.groupby(df["Fecha"].dt.date):
    col_offset = tabla_idx * (len(headers_inv) + len(headers_lim) + len(headers_asi) + 7 + len(headers_inv))
    primera_fila = grupo.iloc[0]
    estaciones_texto = str(primera_fila["Estaciones"]).strip("[]").replace("'", "").replace('"', "")
    estaciones = [x.strip() for x in estaciones_texto.split(",")]

    try:
        asimetria_pura = ast.literal_eval(str(primera_fila["Asimetriamodelo"]))
    except:
        asimetria_pura = []

    tabla = []
    for idx, (_, fila) in enumerate(grupo.iterrows()):
        tabla.append({
            "inv_final": ast.literal_eval(str(fila["Inventario_final"])) if pd.notna(fila["Inventario_final"]) else [],
            "limites": ast.literal_eval(str(fila["Limites"])) if pd.notna(fila["Limites"]) else [],
            "capacidad": ast.literal_eval(str(fila["Capacidad"])) if pd.notna(fila["Capacidad"]) else [],
            "inv_inicial": ast.literal_eval(str(fila["Inventario_inicial"])) if pd.notna(fila["Inventario_inicial"]) else [],
            "idx": idx,
            "s_base": asimetria_pura
        })

    orden = sorted(range(len(estaciones)), key=lambda i: asimetria_pura[i], reverse=True)

    ws2.cell(row=start_row - 2, column=start_col + col_offset, value=str(fecha))
    ws2.cell(row=start_row - 1, column=start_col + col_offset, value="Estacion")
    for i, idx_est in enumerate(orden):
        ws2.cell(row=start_row + i, column=start_col + col_offset, value=estaciones[idx_est])

    for inv_idx in range(len(tabla)):
        col_inv = start_col + col_offset + inv_idx + 1
        ws2.cell(row=start_row - 1, column=col_inv, value=headers_inv[inv_idx % len(headers_inv)])
        for i, idx_est in enumerate(orden):
            if idx_est < len(tabla[inv_idx]["inv_final"]):
                ws2.cell(row=start_row + i, column=col_inv, value=tabla[inv_idx]["inv_final"][idx_est])

    for idx, header in enumerate(headers_asi):
        col_asi_var = start_col + col_offset + len(headers_inv) + 1 + idx
        ws2.cell(row=start_row - 1, column=col_asi_var, value=header)
        for i, idx_est in enumerate(orden):
            if idx_est < len(asimetria_pura):
                valor = asimetria_pura[idx_est] * multipliers[idx]
                ws2.cell(row=start_row + i, column=col_asi_var, value=round(valor, 2))

    col_lim_start = start_col + col_offset + len(headers_inv) + len(headers_asi) + 1
    for lim_idx in range(len(tabla)):
        col_lim = col_lim_start + lim_idx
        ws2.cell(row=start_row - 1, column=col_lim, value=headers_lim[lim_idx % len(headers_lim)])
        for i, idx_est in enumerate(orden):
            if idx_est < len(tabla[lim_idx]["limites"]):
                par = tabla[lim_idx]["limites"][idx_est]
                if isinstance(par, list) and len(par) == 2:
                    valor = f"{round(par[0], 2)}, {round(par[1], 2)}"
                    ws2.cell(row=start_row + i, column=col_lim, value=valor)

    col_binicial = col_lim + 1
    ws2.cell(row=start_row - 1, column=col_binicial, value="B inicial")
    for i, idx_est in enumerate(orden):
        if idx_est < len(tabla[0]["inv_inicial"]):
            ws2.cell(row=start_row + i, column=col_binicial, value=tabla[0]["inv_inicial"][idx_est])

    col_b1900 = col_binicial + 1
    ws2.cell(row=start_row - 1, column=col_b1900, value="B 19:00")
    b_1900_datos_dict = bikes_df[fecha].to_dict() if fecha in bikes_df.columns else {}
    for i, idx_est in enumerate(orden):
        est = estaciones[idx_est]
        valor = b_1900_datos_dict.get(est, None)
        ws2.cell(row=start_row + i, column=col_b1900, value=valor)

    col_mmodelo = col_b1900 + 1
    ws2.cell(row=start_row - 1, column=col_mmodelo, value="Mmodelo")
    for i, idx_est in enumerate(orden):
        if idx_est < len(tabla[0]["capacidad"]):
            ws2.cell(row=start_row + i, column=col_mmodelo, value=tabla[0]["capacidad"][idx_est])

    # === Columnas V ===
    suma_totales = []
    porcentajes_var = []
    for v_idx, v_header in enumerate(v_headers):
        col_v = col_mmodelo + 1 + v_idx
        ws2.cell(row=start_row - 1, column=col_v, value=v_header)

        for i, idx_est in enumerate(orden):
            inv_final_val = tabla[v_idx]["inv_final"][idx_est] if idx_est < len(tabla[v_idx]["inv_final"]) else None
            b_1900_val = ws2.cell(row=start_row + i, column=col_b1900).value
            try:
                b_1900_val = float(b_1900_val)
            except (TypeError, ValueError):
                b_1900_val = None
            val = 0
            if inv_final_val is not None and b_1900_val is not None:
                if i < 10:
                    val = 1 if inv_final_val >= b_1900_val else 0
                else:
                    val = 1 if inv_final_val <= b_1900_val else 0
            ws2.cell(row=start_row + i, column=col_v, value=val)

        suma_1_10 = sum(ws2.cell(row=start_row + r, column=col_v).value or 0 for r in range(0, 10))
        suma_11_20 = sum(ws2.cell(row=start_row + r, column=col_v).value or 0 for r in range(10, 20))
        total_20 = suma_1_10 + suma_11_20
        suma_totales.append(total_20)

        ws2.cell(row=start_row + 20, column=col_v, value=suma_1_10)
        ws2.cell(row=start_row + 21, column=col_v, value=suma_11_20)
        ws2.cell(row=start_row + 22, column=col_v, value=total_20)
        porcentaje = round((total_20 / 20) * 100)
        cell_porcentaje = ws2.cell(row=start_row + 23, column=col_v, value=porcentaje / 100)
        cell_porcentaje.number_format = '0%'
        porcentajes_var.append(porcentaje)

    # === Métricas de resumen por fecha ===
    promedio_total = round(sum(suma_totales) / len(suma_totales), 2)
    std_dev = round(statistics.stdev(suma_totales), 4) if len(suma_totales) >= 2 else 0
    promedio_porcentaje = round(sum(porcentajes_var) / len(porcentajes_var))
    std_dev_porcentaje = round(statistics.stdev(porcentajes_var), 2) if len(porcentajes_var) >= 2 else 0

    col_v50 = col_mmodelo + 1
    ws2.cell(row=start_row + 24, column=col_v50 - 1, value="Promedio")
    ws2.cell(row=start_row + 24, column=col_v50, value=promedio_total)
    ws2.cell(row=start_row + 25, column=col_v50 - 1, value="Desviacion estandar")
    ws2.cell(row=start_row + 25, column=col_v50, value=std_dev)
    ws2.cell(row=start_row + 26, column=col_v50 - 1, value="Promedio %")
    cell_promedio_porcentaje = ws2.cell(row=start_row + 26, column=col_v50, value=promedio_porcentaje / 100)
    cell_promedio_porcentaje.number_format = '0%'
    ws2.cell(row=start_row + 27, column=col_v50 - 1, value="Desv. estándar % variacion")
    ws2.cell(row=start_row + 27, column=col_v50, value=std_dev_porcentaje)

    todos_los_promedios.append(promedio_total)
    todas_las_desviaciones.append(std_dev)
    todos_los_porcentajes.append(promedio_porcentaje)
    desviaciones_porcentaje.append(std_dev_porcentaje)
    # === Promedios de suma_1_10 y suma_11_20 por fecha ===
    promedio_suma_1_10 = round(sum(ws2.cell(row=start_row + 20, column=col_v50 + i).value or 0 for i in range(5)) / 5, 2)
    promedio_suma_11_20 = round(sum(ws2.cell(row=start_row + 21, column=col_v50 + i).value or 0 for i in range(5)) / 5, 2)

    ws2.cell(row=start_row + 28, column=col_v50 - 1, value="Prom. Suma 1-10")
    ws2.cell(row=start_row + 28, column=col_v50, value=promedio_suma_1_10)

    ws2.cell(row=start_row + 29, column=col_v50 - 1, value="Prom. Suma 11-20")
    ws2.cell(row=start_row + 29, column=col_v50, value=promedio_suma_11_20)

    promedios_suma_1_10.append(promedio_suma_1_10)
    promedios_suma_11_20.append(promedio_suma_11_20)

    tabla_idx += 1

# === Promedios globales al final ===
final_row = start_row + 30
ws2.cell(row=final_row, column=1, value="Promedio general conteo")
ws2.cell(row=final_row, column=2, value=round(sum(todos_los_promedios) / len(todos_los_promedios), 2))
ws2.cell(row=final_row + 1, column=1, value="Desviacion estandar promedio")
ws2.cell(row=final_row + 1, column=2, value=round(sum(todas_las_desviaciones) / len(todas_las_desviaciones), 4))
ws2.cell(row=final_row + 2, column=1, value="Promedio general %")
prom_final = int(round(sum(todos_los_porcentajes) / len(todos_los_porcentajes)))
cell_prom_final = ws2.cell(row=final_row + 2, column=2, value=prom_final / 100)
cell_prom_final.number_format = '0%'
ws2.cell(row=final_row + 3, column=1, value="Promedio desv. estándar % variación")
ws2.cell(row=final_row + 3, column=2, value=round(sum(desviaciones_porcentaje) / len(desviaciones_porcentaje), 2))
ws2.cell(row=final_row + 4, column=1, value="Promedio general Suma 1-10")
ws2.cell(row=final_row + 4, column=2, value=round(sum(promedios_suma_1_10) / len(promedios_suma_1_10), 2))

ws2.cell(row=final_row + 5, column=1, value="Promedio general Suma 11-20")
ws2.cell(row=final_row + 5, column=2, value=round(sum(promedios_suma_11_20) / len(promedios_suma_11_20), 2))


# Guardar archivo
wb.save("Resultado_completoapromediocaso2desv.xlsx")

In [2]:
import pandas as pd
from openpyxl import Workbook
import ast
import statistics

# Leer datos
df = pd.read_excel("Resultado_optimizacioncaso1MALfinal.xlsx", sheet_name="Sheet1")
df["Fecha"] = pd.to_datetime(df["Fecha"], errors="coerce")
bikes_df = pd.read_excel("resumen_bikescaso1.xlsx")
bikes_df.columns = [pd.to_datetime(str(col), errors='coerce').date() if isinstance(col, str) else col for col in bikes_df.columns]
estaciones_bikes = bikes_df.iloc[:, 0].tolist()
bikes_df.set_index(bikes_df.columns[0], inplace=True)

# Libro
wb = Workbook()
ws2 = wb.active
ws2.title = "buscarv"

# Parámetros
start_row = 3
start_col = 1
tabla_idx = 0
promedios_suma_1_10 = []
promedios_suma_11_20 = []

# Nuevas listas para V0
v0_sumas_totales = []
v0_porcentajes_totales = []

headers_inv = ["inv final -50%", "inv final -25%", "inv final 0", "inv final +25%", "inv final +50%"]
headers_lim = ["L-50%", "L -25%", "L 0", "L+25%", "L+50%"]
headers_asi = ["S-50", "S-25", "S", "S+25", "S+50"]
v_headers = ["V -50", "V -25", "V 0", "V +25", "V +50"]
multipliers = [0.5, 0.75, 1.0, 1.25, 1.5]

# Totales
todos_los_promedios = []
todas_las_desviaciones = []
todos_los_porcentajes = []
desviaciones_porcentaje = []

for fecha, grupo in df.groupby(df["Fecha"].dt.date):
    col_offset = tabla_idx * (len(headers_inv) + len(headers_lim) + len(headers_asi) + 7 + len(headers_inv))
    primera_fila = grupo.iloc[0]
    estaciones_texto = str(primera_fila["Estaciones"]).strip("[]").replace("'", "").replace('"', "")
    estaciones = [x.strip() for x in estaciones_texto.split(",")]

    try:
        asimetria_pura = ast.literal_eval(str(primera_fila["Asimetriamodelo"]))
    except:
        asimetria_pura = []

    tabla = []
    for idx, (_, fila) in enumerate(grupo.iterrows()):
        tabla.append({
            "inv_final": ast.literal_eval(str(fila["Inventario_final"])) if pd.notna(fila["Inventario_final"]) else [],
            "limites": ast.literal_eval(str(fila["Limites"])) if pd.notna(fila["Limites"]) else [],
            "capacidad": ast.literal_eval(str(fila["Capacidad"])) if pd.notna(fila["Capacidad"]) else [],
            "inv_inicial": ast.literal_eval(str(fila["Inventario_inicial"])) if pd.notna(fila["Inventario_inicial"]) else [],
            "idx": idx,
            "s_base": asimetria_pura
        })

    orden = sorted(range(len(estaciones)), key=lambda i: asimetria_pura[i], reverse=True)

    ws2.cell(row=start_row - 2, column=start_col + col_offset, value=str(fecha))
    ws2.cell(row=start_row - 1, column=start_col + col_offset, value="Estacion")
    for i, idx_est in enumerate(orden):
        ws2.cell(row=start_row + i, column=start_col + col_offset, value=estaciones[idx_est])

    for inv_idx in range(len(tabla)):
        col_inv = start_col + col_offset + inv_idx + 1
        ws2.cell(row=start_row - 1, column=col_inv, value=headers_inv[inv_idx % len(headers_inv)])
        for i, idx_est in enumerate(orden):
            if idx_est < len(tabla[inv_idx]["inv_final"]):
                ws2.cell(row=start_row + i, column=col_inv, value=tabla[inv_idx]["inv_final"][idx_est])

    for idx, header in enumerate(headers_asi):
        col_asi_var = start_col + col_offset + len(headers_inv) + 1 + idx
        ws2.cell(row=start_row - 1, column=col_asi_var, value=header)
        for i, idx_est in enumerate(orden):
            if idx_est < len(asimetria_pura):
                valor = asimetria_pura[idx_est] * multipliers[idx]
                ws2.cell(row=start_row + i, column=col_asi_var, value=round(valor, 2))

    col_lim_start = start_col + col_offset + len(headers_inv) + len(headers_asi) + 1
    for lim_idx in range(len(tabla)):
        col_lim = col_lim_start + lim_idx
        ws2.cell(row=start_row - 1, column=col_lim, value=headers_lim[lim_idx % len(headers_lim)])
        for i, idx_est in enumerate(orden):
            if idx_est < len(tabla[lim_idx]["limites"]):
                par = tabla[lim_idx]["limites"][idx_est]
                if isinstance(par, list) and len(par) == 2:
                    valor = f"{round(par[0], 2)}, {round(par[1], 2)}"
                    ws2.cell(row=start_row + i, column=col_lim, value=valor)

    col_binicial = col_lim + 1
    ws2.cell(row=start_row - 1, column=col_binicial, value="B inicial")
    for i, idx_est in enumerate(orden):
        if idx_est < len(tabla[0]["inv_inicial"]):
            ws2.cell(row=start_row + i, column=col_binicial, value=tabla[0]["inv_inicial"][idx_est])

    col_b1900 = col_binicial + 1
    ws2.cell(row=start_row - 1, column=col_b1900, value="B 19:00")
    b_1900_datos_dict = bikes_df[fecha].to_dict() if fecha in bikes_df.columns else {}
    for i, idx_est in enumerate(orden):
        est = estaciones[idx_est]
        valor = b_1900_datos_dict.get(est, None)
        ws2.cell(row=start_row + i, column=col_b1900, value=valor)

    col_mmodelo = col_b1900 + 1
    ws2.cell(row=start_row - 1, column=col_mmodelo, value="Mmodelo")
    for i, idx_est in enumerate(orden):
        if idx_est < len(tabla[0]["capacidad"]):
            ws2.cell(row=start_row + i, column=col_mmodelo, value=tabla[0]["capacidad"][idx_est])

    # === Columnas V ===
    suma_totales = []
    porcentajes_var = []
    for v_idx, v_header in enumerate(v_headers):
        col_v = col_mmodelo + 1 + v_idx
        ws2.cell(row=start_row - 1, column=col_v, value=v_header)

        for i, idx_est in enumerate(orden):
            inv_final_val = tabla[v_idx]["inv_final"][idx_est] if idx_est < len(tabla[v_idx]["inv_final"]) else None
            b_1900_val = ws2.cell(row=start_row + i, column=col_b1900).value
            try:
                b_1900_val = float(b_1900_val)
            except (TypeError, ValueError):
                b_1900_val = None
            val = 0
            if inv_final_val is not None and b_1900_val is not None:
                if i < 10:
                    val = 1 if inv_final_val >= b_1900_val else 0
                else:
                    val = 1 if inv_final_val <= b_1900_val else 0
            ws2.cell(row=start_row + i, column=col_v, value=val)

        suma_1_10 = sum(ws2.cell(row=start_row + r, column=col_v).value or 0 for r in range(0, 10))
        suma_11_20 = sum(ws2.cell(row=start_row + r, column=col_v).value or 0 for r in range(10, 20))
        total_20 = suma_1_10 + suma_11_20
        suma_totales.append(total_20)

        ws2.cell(row=start_row + 20, column=col_v, value=suma_1_10)
        ws2.cell(row=start_row + 21, column=col_v, value=suma_11_20)
        ws2.cell(row=start_row + 22, column=col_v, value=total_20)
        porcentaje = round((total_20 / 20) * 100)
        cell_porcentaje = ws2.cell(row=start_row + 23, column=col_v, value=porcentaje / 100)
        cell_porcentaje.number_format = '0%'
        porcentajes_var.append(porcentaje)

        # Extraer valores solo si es la columna V 0
        if v_idx == 2:
            v0_sumas_totales.append(total_20)
            v0_porcentajes_totales.append(porcentaje)

    # === Métricas de resumen por fecha ===
    promedio_total = round(sum(suma_totales) / len(suma_totales), 2)
    std_dev = round(statistics.stdev(suma_totales), 4) if len(suma_totales) >= 2 else 0
    promedio_porcentaje = round(sum(porcentajes_var) / len(porcentajes_var))
    std_dev_porcentaje = round(statistics.stdev(porcentajes_var), 2) if len(porcentajes_var) >= 2 else 0

    col_v50 = col_mmodelo + 1
    ws2.cell(row=start_row + 24, column=col_v50 - 1, value="Promedio")
    ws2.cell(row=start_row + 24, column=col_v50, value=promedio_total)
    ws2.cell(row=start_row + 25, column=col_v50 - 1, value="Desviacion estandar")
    ws2.cell(row=start_row + 25, column=col_v50, value=std_dev)
    ws2.cell(row=start_row + 26, column=col_v50 - 1, value="Promedio %")
    cell_promedio_porcentaje = ws2.cell(row=start_row + 26, column=col_v50, value=promedio_porcentaje / 100)
    cell_promedio_porcentaje.number_format = '0%'
    ws2.cell(row=start_row + 27, column=col_v50 - 1, value="Desv. estándar % variacion")
    ws2.cell(row=start_row + 27, column=col_v50, value=std_dev_porcentaje)

    todos_los_promedios.append(promedio_total)
    todas_las_desviaciones.append(std_dev)
    todos_los_porcentajes.append(promedio_porcentaje)
    desviaciones_porcentaje.append(std_dev_porcentaje)

    promedio_suma_1_10 = round(sum(ws2.cell(row=start_row + 20, column=col_v50 + i).value or 0 for i in range(5)) / 5, 2)
    promedio_suma_11_20 = round(sum(ws2.cell(row=start_row + 21, column=col_v50 + i).value or 0 for i in range(5)) / 5, 2)

    ws2.cell(row=start_row + 28, column=col_v50 - 1, value="Prom. Suma 1-10")
    ws2.cell(row=start_row + 28, column=col_v50, value=promedio_suma_1_10)
    ws2.cell(row=start_row + 29, column=col_v50 - 1, value="Prom. Suma 11-20")
    ws2.cell(row=start_row + 29, column=col_v50, value=promedio_suma_11_20)

    promedios_suma_1_10.append(promedio_suma_1_10)
    promedios_suma_11_20.append(promedio_suma_11_20)

    tabla_idx += 1

# === Promedios globales al final ===
final_row = start_row + 30
ws2.cell(row=final_row, column=1, value="Promedio general conteo")
ws2.cell(row=final_row, column=2, value=round(sum(todos_los_promedios) / len(todos_los_promedios), 2))
ws2.cell(row=final_row + 1, column=1, value="Desviacion estandar promedio")
ws2.cell(row=final_row + 1, column=2, value=round(sum(todas_las_desviaciones) / len(todas_las_desviaciones), 4))
ws2.cell(row=final_row + 2, column=1, value="Promedio general %")
prom_final = int(round(sum(todos_los_porcentajes) / len(todos_los_porcentajes)))
cell_prom_final = ws2.cell(row=final_row + 2, column=2, value=prom_final / 100)
cell_prom_final.number_format = '0%'
ws2.cell(row=final_row + 3, column=1, value="Promedio desv. estándar % variación")
ws2.cell(row=final_row + 3, column=2, value=round(sum(desviaciones_porcentaje) / len(desviaciones_porcentaje), 2))
ws2.cell(row=final_row + 4, column=1, value="Promedio general Suma 1-10")
ws2.cell(row=final_row + 4, column=2, value=round(sum(promedios_suma_1_10) / len(promedios_suma_1_10), 2))
ws2.cell(row=final_row + 5, column=1, value="Promedio general Suma 11-20")
ws2.cell(row=final_row + 5, column=2, value=round(sum(promedios_suma_11_20) / len(promedios_suma_11_20), 2))

# === Nuevos cálculos globales para V0 ===
ws2.cell(row=final_row + 6, column=1, value="Promedio general V0")
ws2.cell(row=final_row + 6, column=2, value=round(sum(v0_sumas_totales) / len(v0_sumas_totales), 2))

ws2.cell(row=final_row + 7, column=1, value="Promedio de % V0")
ws2.cell(row=final_row + 7, column=2, value=round(sum(v0_porcentajes_totales) / len(v0_porcentajes_totales) / 100, 4))

ws2.cell(row=final_row + 8, column=1, value="Desviación estándar % V0")
ws2.cell(row=final_row + 8, column=2, value=round(statistics.stdev(v0_porcentajes_totales) / 100, 4) if len(v0_porcentajes_totales) >= 2 else 0)

# Guardar archivo
wb.save("Resultado_completoapromediocaso1todasdesvintetoPREOCUPA.xlsx")
